# GPU Benchmark Datasets Runner

这个 notebook 将 `gpu/sanity_check/v5_mat32_for_precompute.ipynb` 里的 GPU 算法整理为一个真实数据集 benchmark 入口。

支持的目标：

- 只发现并加载 `gpu/benchmark_dataset/processed` 下的标准化数据集
- 每个原始数据集由单独的预处理 `py` 脚本负责清洗、投影、切分和标准化
- notebook 只负责实验配置、训练、统计和可选 SLQ
- 将 `kernel`、`eps`、`MODE_SPECS`、`PRECOMPUTE_METHODS=["original", "C1"]` 做成可调 config
- 对比 `gpu_v3_topq` 与 `gpu_v3_topq_eigenpro_nystrom`
- 保留原有 Colab / GitHub scaffold

当前 `3D_spatial_network` 的预处理脚本为：`preprocess_3d_spatial_network.py`。

标准处理后数据集约定：

- `x_train`, `x_test`, `y_train`, `y_test` 保存在 `.npz`
- 可选 sidecar `.json` 保存投影、缩放、切分、清洗等 metadata
- notebook 仅导入这些处理好的数组，不再直接读取原始表格

In [ ]:
'''
# ---- Optional Colab / Drive setup ----
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
DRIVE_MOUNT_DIR = Path("/content/drive")
DRIVE_MYDRIVE_DIR = DRIVE_MOUNT_DIR / "MyDrive"
DRIVE_CACHE_ENABLED = True
DRIVE_CACHE_SEARCH_RECURSIVE = True
DRIVE_CACHE_CANDIDATE_DIRS = [
    DRIVE_MYDRIVE_DIR / "EFGP_Eigenpro" / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR / "Colab_Experiments" / "EFGP_Eigenpro" / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR / "benchmark_dataset_cache",
    DRIVE_MYDRIVE_DIR,
]

if IS_COLAB:
    try:
        from google.colab import drive
        if not DRIVE_MYDRIVE_DIR.exists():
            drive.mount(str(DRIVE_MOUNT_DIR))
    except Exception as e:
        print("Drive mount skipped:", e)

print("IS_COLAB:", IS_COLAB)
print("DRIVE_MYDRIVE_DIR:", DRIVE_MYDRIVE_DIR)
print("DRIVE_CACHE_ENABLED:", DRIVE_CACHE_ENABLED)
'''

In [ ]:
## For github import
'''
import os
import sys

GITHUB_USER = "Yifiwifi"
REPO_NAME = "EFGP-Eigenpro"
SUB_DIR = "efgp_eigenpro_py"
PROJECT_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(PROJECT_PATH):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
else:
    %cd {PROJECT_PATH}
    !git pull origin main

CODE_ROOT = os.path.join(PROJECT_PATH, SUB_DIR)
if CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

print("Checking runtime dependencies")
!pip install cufinufft cupy-cuda12x --extra-index-url https://pypi.nvidia.com

requirements_path = os.path.join(CODE_ROOT, "requirements.txt")
if os.path.exists(requirements_path):
    !pip install -r {requirements_path}

benchmark_dir_path = os.path.join(CODE_ROOT, "gpu/benchmark_dataset")
if os.path.exists(benchmark_dir_path):
    os.chdir(benchmark_dir_path)
    print("cwd:", os.getcwd())
else:
    print("benchmark_dataset path not found:", benchmark_dir_path)

# Refresh runtime library path for some Colab images
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
!ldconfig /usr/local/lib

print("=" * 40)
try:
    import torch
    import cupy as cp
    import cufinufft
    cp.cuda.Stream.null.synchronize()
    print("PyTorch:", torch.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    print("cufinufft import ok")
except Exception as e:
    print("runtime check failed:", e)
print("=" * 40)
'''

In [ ]:
import gc
import json
import math
import os
import re
import sys
import time
import traceback
import itertools
import importlib
from dataclasses import asdict, is_dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

BENCHMARK_DIR = None
for p in (_here, *_here.parents):
    cand = p / "efgp_eigenpro_py" / "gpu" / "benchmark_dataset"
    if cand.exists():
        BENCHMARK_DIR = cand
        break
if BENCHMARK_DIR is None:
    BENCHMARK_DIR = Path("D:/NU/ML/efgp_eigenpro_py/gpu/benchmark_dataset").resolve()

from efgp_eigenpro_py.kernels import make_matern, make_squared_exponential
from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.discretization import basis_weights, choose_grid_params
from efgp_eigenpro_py.gpu.backends import BackendConfig, build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import ensure_gpu_data_context
from efgp_eigenpro_py.gpu.versions import GPURunConfig, run_v1_pure_efgp, run_v3_full_gpu_eigenspace
from efgp_eigenpro_py.gpu.v3_eigenspace import EigenspaceConfig
from efgp_eigenpro_py.gpu.v1_ops import _device_array_to_numpy, predict_v1

import efgp_eigenpro_py.gpu as _gpu_pkg_bm
import efgp_eigenpro_py.gpu.v1_ops as _gpu_v1_ops_bm
import efgp_eigenpro_py.gpu.versions as _gpu_versions_bm
import efgp_eigenpro_py.gpu.binned_efgp_precompute as _binned_pc_mod

try:
    import cupy as cp
except Exception:
    cp = None

np.set_printoptions(precision=6, suppress=True)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])
print("benchmark dir:", BENCHMARK_DIR)
print("cupy available:", cp is not None)

# Dataset 

In [ ]:
# ---- Dataset discovery preview ----
RAW_DATA_DIR = BENCHMARK_DIR
PROCESSED_DATA_DIR = BENCHMARK_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_SUFFIXES = (".npz",)


def discover_processed_datasets(data_dir: Path, suffixes=PROCESSED_DATA_SUFFIXES) -> list[Path]:
    return sorted(
        [p for p in data_dir.iterdir() if p.is_file() and p.suffix.lower() in suffixes],
        key=lambda p: p.name.lower(),
    )


PREVIEW_DISCOVERED_DATASET_FILES = discover_processed_datasets(PROCESSED_DATA_DIR)
PREVIEW_AVAILABLE_DATASET_NAMES = sorted([p.stem for p in PREVIEW_DISCOVERED_DATASET_FILES])
print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("PREVIEW_AVAILABLE_DATASET_NAMES:", PREVIEW_AVAILABLE_DATASET_NAMES)


In [ ]:
# ---- Dataset selection ----
# 规则：写 stem 名称即可（也就是 processed/*.npz 去掉 .npz 的部分）
# - RUN_ALL_DATASETS=True  : 跑 processed/ 下发现到的全部数据集
# - RUN_ALL_DATASETS=False : 跑 DATASET_SELECTION_LIST 里列出的子集（注释/取消注释即可）
RUN_ALL_DATASETS = False

# ---- Synthetic (true func 2D) ----
# 你已在 Drive 缓存里准备了这些 N_train 的 npz；这里用它们批量生成 stem。
N_TRAIN_LIST = [
    100_000,
    300_000,
    1_000_000,
    3_000_000,
    10_000_000,
    30_000_000,
    100_000_000,
]

# 旧的单点 synthetic 配置（保留以便回滚）
# SYNTHETIC_N_TRAIN = 10_000_000
# SYNTHETIC_EXPECTED_STEM = f"synthetic_true_func_2d_n{int(SYNTHETIC_N_TRAIN)}"

SYNTHETIC_DATASET_STEMS = [f"synthetic_true_func_2d_n{int(n)}" for n in N_TRAIN_LIST]

# ---- USGS / USGC lidar ----
# 对照 preprocess_usgs_3dep_lidar_size_sweep.py：它生成的 stem 形如 "<prefix>_ntrain{N}"。
# 你的 processed/ 目录里也能看到这套命名（..._ntrain100000 等）。
USGS_BASE_STEM = "USGS_LPC_IL_Winnebago_2018_ground_elevation_regression"

# 推荐：使用 ntrain 命名（与 size_sweep 脚本一致）
USGS_DATASET_STEMS = [f"{USGS_BASE_STEM}_ntrain{int(n)}" for n in N_TRAIN_LIST]

# 备选：旧命名（如果你的 Drive 里是 ..._n{N}，就把上面一行注释、启用下面这行）
# USGS_DATASET_STEMS = [f"{USGS_BASE_STEM}_n{int(n)}" for n in N_TRAIN_LIST]

USGS_AUTO_PREPROCESS_ARGS = []  # 仅在你明确启用 USGS 自动预处理时才会用到

# 你描述里写的是 USGC，这里做别名，指向同一套 stem。
USGC_BASE_STEM = USGS_BASE_STEM
USGC_DATASET_STEMS = list(USGS_DATASET_STEMS)

DATASET_SELECTION_LIST = [
    # 可先参考上一格输出的 PREVIEW_AVAILABLE_DATASET_NAMES 再决定勾选哪些数据集：
    # "3D_spatial_network_utm32_altitude_regression",
    # "household_power_consumption_global_active_power_d1_time",

    # Synthetic sweep (keep all N_train in N_TRAIN_LIST)
    *SYNTHETIC_DATASET_STEMS,

    # USGS / USGC sweep (same N_train list)
    *USGS_DATASET_STEMS,

    # 如需跑“非 sweep 的 base 版本”，可打开这行（前提是 processed/ 或 Drive 里确实有该 stem 的 npz）
    # USGS_BASE_STEM,
]
SELECTED_DATASET_STEMS = {Path(str(x)).stem for x in DATASET_SELECTION_LIST}


In [ ]:
# ---- Dataset auto preprocess / refresh ----
# 行为：优先从 /content/drive/MyDrive 下恢复缓存的 npz/json 到本地 processed/；
# 如果恢复后仍缺文件，才自动运行对应预处理脚本。
# 如需强制重新生成，可把 *_AUTO_PREPROCESS_FORCE 设为 True。
import shutil
import subprocess

# ---- Drive cache defaults (auto-enable on Colab) ----
# 目标：如果 Drive 上存在对应 stem 的 .npz/.json，就自动恢复到本地 processed/，避免重复生成。
# 若不在 Colab，则默认关闭 Drive cache（本地运行不需要挂载 Drive）。
IS_COLAB = "google.colab" in sys.modules
if "DRIVE_CACHE_ENABLED" not in globals():
    DRIVE_MOUNT_DIR = Path("/content/drive")
    DRIVE_MYDRIVE_DIR = DRIVE_MOUNT_DIR / "MyDrive"
    DRIVE_CACHE_ENABLED = bool(IS_COLAB)
    DRIVE_CACHE_SEARCH_RECURSIVE = True
    DRIVE_CACHE_CANDIDATE_DIRS = [
        DRIVE_MYDRIVE_DIR / "EFGP_Eigenpro" / "benchmark_dataset_cache",
        DRIVE_MYDRIVE_DIR / "Colab_Experiments" / "EFGP_Eigenpro" / "benchmark_dataset_cache",
        DRIVE_MYDRIVE_DIR / "benchmark_dataset_cache",
        DRIVE_MYDRIVE_DIR,
    ]

if IS_COLAB and DRIVE_CACHE_ENABLED:
    try:
        from google.colab import drive
        if not DRIVE_MYDRIVE_DIR.exists():
            drive.mount(str(DRIVE_MOUNT_DIR))
    except Exception as e:
        print("Drive mount skipped:", e)

SYNTHETIC_AUTO_PREPROCESS_FORCE = False

# USGS/USGC 数据如果你已经放在 Drive cache，通常不希望 notebook 自动去下载/重建。
# 默认关闭自动预处理；只做 Drive restore。需要自动生成时再手动改 True。
USGS_AUTO_PREPROCESS_ENABLED = False
USGS_AUTO_PREPROCESS_FORCE = False

# 这里的 stem 列表来自上一格（只保留 synthetic + USGS + N_TRAIN_LIST）。
SYNTHETIC_SELECTED_IN_LIST = any(stem in SELECTED_DATASET_STEMS for stem in SYNTHETIC_DATASET_STEMS)
USGS_SELECTED_IN_LIST = any(stem in SELECTED_DATASET_STEMS for stem in USGS_DATASET_STEMS)


def _copy_file_if_missing(src: Path, dst: Path) -> bool:
    if dst.exists() or not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True


def _find_drive_cached_file(filename: str) -> Path | None:
    if not bool(DRIVE_CACHE_ENABLED):
        return None
    for base in DRIVE_CACHE_CANDIDATE_DIRS:
        if base.exists():
            direct = base / filename
            if direct.exists():
                return direct
    if not bool(DRIVE_CACHE_SEARCH_RECURSIVE) or not DRIVE_MYDRIVE_DIR.exists():
        return None
    try:
        for p in DRIVE_MYDRIVE_DIR.rglob(filename):
            if p.is_file():
                return p
    except Exception as e:
        print(f"drive cache search skipped for {filename}: {e}")
    return None


def restore_dataset_from_drive_cache(dataset_stem: str) -> dict:
    restored = {"npz": False, "json": False, "from": {}}
    for suffix in (".npz", ".json"):
        filename = f"{dataset_stem}{suffix}"
        dst = PROCESSED_DATA_DIR / filename
        src = _find_drive_cached_file(filename)
        if src is None:
            continue
        copied = _copy_file_if_missing(src, dst)
        restored[suffix[1:]] = bool(copied or dst.exists())
        restored["from"][suffix[1:]] = str(src)
        if copied:
            print(f"restored from Drive cache: {dst.name} <- {src}")
    return restored


def maybe_restore_selected_datasets_from_drive() -> None:
    if RUN_ALL_DATASETS:
        return
    for dataset_stem in sorted(SELECTED_DATASET_STEMS):
        restore_dataset_from_drive_cache(dataset_stem)


maybe_restore_selected_datasets_from_drive()


def maybe_preprocess_synthetic_datasets() -> None:
    if RUN_ALL_DATASETS or not SYNTHETIC_SELECTED_IN_LIST:
        return
    script_path = BENCHMARK_DIR / "preprocess_synthetic_true_func_2d.py"
    if not script_path.exists():
        raise FileNotFoundError(f"Synthetic preprocess script not found: {script_path}")

    for n_train, dataset_stem in zip(N_TRAIN_LIST, SYNTHETIC_DATASET_STEMS):
        if dataset_stem not in SELECTED_DATASET_STEMS:
            continue
        output_npz = PROCESSED_DATA_DIR / f"{dataset_stem}.npz"
        output_json = PROCESSED_DATA_DIR / f"{dataset_stem}.json"
        if not SYNTHETIC_AUTO_PREPROCESS_FORCE and output_npz.exists() and output_json.exists():
            continue
        cmd = [
            sys.executable,
            str(script_path),
            "--n-train",
            str(int(n_train)),
            "--dataset-stem",
            str(dataset_stem),
        ]
        print("running preprocess:", " ".join(cmd))
        subprocess.run(cmd, check=True, cwd=str(BENCHMARK_DIR))
        print("Synthetic preprocess done:", output_npz)


def maybe_preprocess_usgs_datasets() -> None:
    # 默认不自动预处理 USGS（避免在 Colab 上意外触发下载/重建）。
    # 期望行为：优先从 Drive cache restore；若仍缺失，就直接报缺文件。
    if not bool(USGS_AUTO_PREPROCESS_ENABLED):
        return
    if RUN_ALL_DATASETS or not USGS_SELECTED_IN_LIST:
        return

    script_path = BENCHMARK_DIR / "preprocess_usgs_3dep_lidar_size_sweep.py"
    if not script_path.exists():
        raise FileNotFoundError(f"USGS preprocess script not found: {script_path}")

    # 如果所有选中的 USGS stem 都已存在（例如已从 Drive restore），则不做任何事。
    missing = []
    for stem in USGS_DATASET_STEMS:
        if stem not in SELECTED_DATASET_STEMS:
            continue
        npz_path = PROCESSED_DATA_DIR / f"{stem}.npz"
        json_path = PROCESSED_DATA_DIR / f"{stem}.json"
        if not (npz_path.exists() and json_path.exists()):
            missing.append(stem)

    if (not USGS_AUTO_PREPROCESS_FORCE) and (len(missing) == 0):
        return

    n_train_csv = ",".join(str(int(n)) for n in N_TRAIN_LIST)
    cmd = [
        sys.executable,
        str(script_path),
        "--n-train-list",
        n_train_csv,
        "--dataset-stem-prefix",
        str(USGS_BASE_STEM),
        *USGS_AUTO_PREPROCESS_ARGS,
    ]
    print("running preprocess:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(BENCHMARK_DIR))
    print("USGS preprocess done (size sweep):", PROCESSED_DATA_DIR)


maybe_preprocess_synthetic_datasets()
maybe_preprocess_usgs_datasets()

DISCOVERED_DATASET_FILES = discover_processed_datasets(PROCESSED_DATA_DIR)
# Use stem as the stable dataset id (so selection can omit the .npz extension).
DISCOVERED_DATASET_MAP = {p.stem: p for p in DISCOVERED_DATASET_FILES}
AVAILABLE_DATASET_NAMES = sorted(list(DISCOVERED_DATASET_MAP.keys()))

if RUN_ALL_DATASETS:
    DATASET_SELECTION = AVAILABLE_DATASET_NAMES[:]
else:
    # 允许你在列表里写带扩展名的名字，这里统一做 stem 规范化
    DATASET_SELECTION = [Path(str(x)).stem for x in DATASET_SELECTION_LIST]

# Start

In [ ]:


# ---- Kernel / solver sweep ----
KERNEL_SPECS = [
  
    {
        "name": "mat32_ls0.1",
        "family": "matern",
        "nu": 1.5,
        "lengthscale": 0.1,
        "variance": 1.0,
    },
]

'''
        {
        "name": "gaussian_ls0.1",
        "family": "gaussian",
        "lengthscale": 0.1,
        "variance": 1.0,
    },
'''


EPS_LIST = [1e-5]

# ---- Eigenspace method selection moved to next cell ----
# 下面这一格只保留 kernel / solver / benchmark 的通用设置；
# 所有 eigenspace 方法开关、参数和 MODE_SPECS 统一放到下一格。
PRECOMPUTE_METHODS = ["original", "C1"]
PRECOMPUTE_C1_MIN_N_TOTAL = 1_000_000  # None 表示无阈值；仅当数据集总样本数 >= 该值时才启用 C1

# ---- Coordinate Nyström gate (EigenPro Nyström) ----
# M = mtot^dim where mtot comes from choose_grid_params(kernel, eps, L).
# Only enable coordinate_nystrom when M > threshold.
# Default threshold is set to M(mat32_ls0.1, eps=1e-7) on 3D_spatial_network: M=480249 (mtot=693, dim=2).
EIGENPRO_COORD_NYSTROM_MIN_M = 200_000  # None 表示不限制

REPEATS = 1

REG_LAMBDA = 0.1
SOLVE_TOL = 1e-6
GPU_MAXITER = 3000
GPU_NUFFT = "auto"
L2_SCALED = True
DEBUG_FINITE_CHECKS = False

RUN_WARMUP = True
WARMUP_TRAIN_SAMPLES = 10_000

# ---- Binned GPU precompute ----
BINNED_QUALITY = "balanced"
BINNED_USE_SPARSE_BINS = False
BINNED_USE_GPU_DENSE_BINS = True
BINNED_ALLOW_EXACT_NUFFT_FALLBACK = False
BINNED_NUFFT_ALLOW_CPU_FALLBACK = False
BINNED_R_USER = None
BENCHMARK_AFTER_CASE_GPU_POOL_FLUSH = True

# ---- EigenPro Nyström defaults ----
# 这些是全局默认值；会被 MODE_SPECS 中每个 nystrom case 的字段覆盖（如 nystrom_precond_kind / nystrom_refine_mode）。
EIGENPRO_NYSTROM_PRECOND_KIND = "coordinate_nystrom"
EIGENPRO_NYSTROM_REFINE_MODE = "auto"
EIGENPRO_NYSTROM_SURROGATE_SIZE =1600
EIGENPRO_NYSTROM_LOWFREQ_RATIO = 0.5
EIGENPRO_NYSTROM_OVERSAMPLE = 10
EIGENPRO_NYSTROM_RITZ_REFINE = True
EIGENPRO_NYSTROM_SEED = 0
EIGENPRO_NYSTROM_BLOCK_ROWS = 8192
EIGENPRO_NYSTROM_RITZ_BLOCK_COLS = 16
EIGENPRO_NYSTROM_LIFT =  True
EIGENPRO_NYSTROM_REFINE_ITERS = 1


# Damping for coordinate Nyström preconditioner:
#   P_{S,gamma}(v) = v - gamma * I_S V diag(1 - mu/theta) V^* v[S]
# Try gamma in [0.1, 0.25, 0.5, 1.0].
EIGENPRO_COORD_NYSTROM_GAMMA = 1.0

V3_OVERSAMPLE = 16
V3_N_ITER = 3

# ---- Optional SLQ ----
ENABLE_SLQ = False

# 默认 SLQ 策略：
# - 只跑 baseline q=0 + 默认 V3 的最大 q
# - 每个数据集家族（synthetic / usgs）只取最小 N_train 的一个数据集
# 如需手动指定，直接填写 SLQ_SELECTED_MODE_SPECS / SLQ_SELECTED_DATASETS 即可覆盖。
SLQ_USE_DEFAULT_V3_ONLY = True
SLQ_USE_MIN_N_PER_FAMILY = True

SLQ_SELECTED_MODE_SPECS = []   # 留空时：若上面开关为 True，则自动选 q=0 + V3 max-q；否则沿用 MODE_SPECS（仍会自动跳过 eigenpro_nystrom）
SLQ_SELECTED_DATASETS = []     # 留空时：若上面开关为 True，则自动选每个 family 的最小 N；否则沿用 DATASET_SELECTION
SLQ_SELECTED_PRECOMPUTE_METHOD = "original"
SLQ_CFG_KWARGS = {
    "nv": 32,
    "k_max": 300,
    "hermitian_type": "complex",
    "seed": 0,
    "breakdown_abs_tol": 1e-14,
    "breakdown_rel_tol": 1e-12,
    "reorth_mode": "none",
    "reorth_window": 8,
    "reorth_passes": 2,
    "sync_timing": True,
}
SLQ_SUMMARY_MODE = "spd"
SLQ_PREFIX_STEP = 20
SLQ_SEED_BASE = 99173

# ---- Outputs ----
RUN_TAG = datetime.now().strftime("gpu_dataset_benchmark_%Y%m%d_%H%M%S")
OUT_DIR = BENCHMARK_DIR / "outputs" / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_OUTPUTS_DIR = OUT_DIR / "datasets"
DATASET_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_RAW_CSV = OUT_DIR / "global_raw_runs.csv"
GLOBAL_SUMMARY_CSV = OUT_DIR / "global_summary.csv"
GLOBAL_ENV_JSON = OUT_DIR / "global_env_info.json"

print("RAW_DATA_DIR:", RAW_DATA_DIR)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("AVAILABLE_DATASET_NAMES:", AVAILABLE_DATASET_NAMES)
print("DATASET_SELECTION:", DATASET_SELECTION)
print("PRECOMPUTE_METHODS:", PRECOMPUTE_METHODS)
print("PRECOMPUTE_C1_MIN_N_TOTAL:", PRECOMPUTE_C1_MIN_N_TOTAL)
print("EIGENPRO_COORD_NYSTROM_MIN_M:", EIGENPRO_COORD_NYSTROM_MIN_M)
print("EPS_LIST:", EPS_LIST)
print("OUT_DIR:", OUT_DIR)
print("DATASET_OUTPUTS_DIR:", DATASET_OUTPUTS_DIR)

In [ ]:
# ---- Eigenspace methods / MODE_SPECS ----
# 统一把 eigenspace 相关开关放在这一格，便于集中开关和调参。
# 每个方法都可以单独启用/关闭；真正跑哪些 case 由 MODE_SPECS 自动拼出来。

V3_TOPQ_LIST = [ 45, 90, 135, 180]
NYSTROM_TOPQ_LIST = [45, 90, 135, 180]
EXTRA_TOPQ_LIST = V3_TOPQ_LIST

EIG_METHOD_TOGGLES = {
    # baselines
    "baseline_v1_topq0": True,
    "baseline_v3_topq": True,
    # EigenPro Nyström main
    "nystrom_compact_coordinate": True,
    "nystrom_matvec_lift": True,
    #"nystrom_krylov_ritz": True,
    "nystrom_subspace_polish": True,
    # EigenPro Nyström levels
    "nystrom_adaptive_support": True,
    #"nystrom_diag_adaptive_support": True,
    #"nystrom_hybrid_topr": True,
    # EigenPro Nyström ablations
    #"nystrom_inject": False,
    "nystrom_toeplitz_lift": False,
    # extra eigenspace algorithms
    #"extra_ensemble_coordinate_nystrom": True,
    "extra_rand_range_onepass": True,
    #"extra_random_support_lift": True,
    #"extra_chebyshev_filtered_subspace": True,
}


def _eig_enabled(name: str) -> bool:
    return bool(EIG_METHOD_TOGGLES.get(str(name), False))


# ---- Extra eigenspace algorithm params ----
EXTRA_RAND_RANGE_OVERSAMPLE = 16
EXTRA_RAND_RANGE_POWER_ITERS = 0
EXTRA_RAND_RANGE_OMEGA_KIND = "gaussian"  # "gaussian" / "rademacher" / "sparse"
EXTRA_RAND_RANGE_BLOCK_COLS = 16

EXTRA_CHEBY_DEGREE = 4
EXTRA_CHEBY_LAMBDA_LOW = REG_LAMBDA 
EXTRA_CHEBY_LAMBDA_CUT = 1000
EXTRA_CHEBY_OVERSAMPLE = 16
EXTRA_CHEBY_BLOCK_COLS = 16

EXTRA_ENSEMBLE_S = 1600
EXTRA_ENSEMBLE_Q_EACH = 8
EXTRA_ENSEMBLE_N_SKETCHES = 16
EXTRA_ENSEMBLE_GAMMA = EIGENPRO_COORD_NYSTROM_GAMMA

EXTRA_SUPPORT_LIFT_S = 1600
EXTRA_SUPPORT_LIFT_Q_EACH = 8
EXTRA_SUPPORT_LIFT_N_SKETCHES = 16
EXTRA_SUPPORT_LIFT_R_FULL = None  # None => auto from top_q
EXTRA_SUPPORT_LIFT_BLOCK_COLS = 16

# ---- EigenPro Nyström params ----
EIGENPRO_NYSTROM_HYBRID_TOP_R = None  # None => default r = floor(q/4)

EIGENPRO_NYSTROM_MAIN_VARIANTS = [
    {"enabled": _eig_enabled("nystrom_compact_coordinate"), "name": "compact_coordinate", "precond_kind": "coordinate_nystrom", "refine_mode": None},
    {"enabled": _eig_enabled("nystrom_matvec_lift"), "name": "matvec_lift", "precond_kind": "original", "refine_mode": "matvec_lift"},
    {"enabled": _eig_enabled("nystrom_krylov_ritz"), "name": "krylov_ritz", "precond_kind": "original", "refine_mode": "krylov_ritz"},
    {"enabled": _eig_enabled("nystrom_subspace_polish"), "name": "subspace_polish", "precond_kind": "original", "refine_mode": "subspace_polish"},
]

EIGENPRO_NYSTROM_LEVELS_VARIANTS = [
    {"enabled": _eig_enabled("nystrom_adaptive_support"), "name": "adaptive_support", "precond_kind": "original", "refine_mode": "adaptive_support"},
    {"enabled": _eig_enabled("nystrom_diag_adaptive_support"), "name": "diag_adaptive_support", "precond_kind": "original", "refine_mode": "diag_adaptive_support"},
    {
        "enabled": _eig_enabled("nystrom_hybrid_topr"),
        "name": "hybrid_topr",
        "precond_kind": "original",
        "refine_mode": "hybrid_topr",
        "method_cfg": ({} if EIGENPRO_NYSTROM_HYBRID_TOP_R is None else {"hybrid_top_r": int(EIGENPRO_NYSTROM_HYBRID_TOP_R)}),
    },
]

EIGENPRO_NYSTROM_ABLATION_VARIANTS = [
    {"enabled": _eig_enabled("nystrom_inject"), "name": "inject", "precond_kind": "original", "refine_mode": "inject"},
    {"enabled": _eig_enabled("nystrom_toeplitz_lift"), "name": "toeplitz_lift", "precond_kind": "original", "refine_mode": "toeplitz_lift"},
]

# 保留这几个兼容变量，方便 notebook 其他位置继续读到
EIGENPRO_NYSTROM_INCLUDE_LEVELS = any(bool(v.get("enabled", False)) for v in EIGENPRO_NYSTROM_LEVELS_VARIANTS)
EIGENPRO_NYSTROM_INCLUDE_ABLATION = any(bool(v.get("enabled", False)) for v in EIGENPRO_NYSTROM_ABLATION_VARIANTS)
INCLUDE_EXTRA_EIG_METHODS = any(
    _eig_enabled(k)
    for k in (
        "extra_ensemble_coordinate_nystrom",
        "extra_rand_range_onepass",
        "extra_random_support_lift",
        "extra_chebyshev_filtered_subspace",
    )
)


def _active_variants(variants):
    return [dict(v) for v in variants if bool(v.get("enabled", False))]


def _build_nystrom_mode_specs(top_q_list, variants):
    specs = []
    for q in top_q_list:
        tq = int(q)
        if tq <= 0:
            raise ValueError(f"top_q must be > 0 for nystrom modes, got {q}")
        for v in variants:
            name = str(v.get("name", "custom"))
            specs.append(
                {
                    "mode": "gpu_v3_topq_eigenpro_nystrom",
                    "method_variant": name,
                    "top_q": tq,
                    "nystrom_variant": name,
                    "nystrom_precond_kind": str(v.get("precond_kind") or "coordinate_nystrom").lower(),
                    "nystrom_refine_mode": v.get("refine_mode", None),
                    "nystrom_refine_iters": v.get("refine_iters", None),
                    "nystrom_method_cfg": dict(v.get("method_cfg") or {}),
                }
            )
    return specs


def _build_extra_eig_mode_specs(top_q_list):
    specs = []
    for q in top_q_list:
        tq = int(q)
        if tq <= 0:
            raise ValueError(f"top_q must be > 0 for extra eigenspace modes, got {q}")

        if _eig_enabled("extra_ensemble_coordinate_nystrom"):
            specs.append(
                {
                    "mode": "gpu_v3_custom_topq",
                    "method_variant": "ensemble_coordinate_nystrom",
                    "top_q": tq,
                    "eig_method": "ensemble_coordinate_nystrom",
                    "method_cfg": {
                        "s": int(EXTRA_ENSEMBLE_S),
                        "q_each": int(EXTRA_ENSEMBLE_Q_EACH),
                        "n_sketches": int(EXTRA_ENSEMBLE_N_SKETCHES),
                        "gamma": float(EXTRA_ENSEMBLE_GAMMA),
                    },
                }
            )

        if _eig_enabled("extra_rand_range_onepass"):
            specs.append(
                {
                    "mode": "gpu_v3_custom_topq",
                    "method_variant": "rand_range_onepass",
                    "top_q": tq,
                    "eig_method": "rand_range_onepass",
                    "method_cfg": {
                        "oversample": int(EXTRA_RAND_RANGE_OVERSAMPLE),
                        "power_iters": int(EXTRA_RAND_RANGE_POWER_ITERS),
                        "omega_kind": str(EXTRA_RAND_RANGE_OMEGA_KIND),
                        "block_cols": int(EXTRA_RAND_RANGE_BLOCK_COLS),
                    },
                }
            )

        if _eig_enabled("extra_random_support_lift"):
            r_full_eff = (
                int(max(tq, min(4 * tq, tq + 16)))
                if EXTRA_SUPPORT_LIFT_R_FULL is None
                else int(EXTRA_SUPPORT_LIFT_R_FULL)
            )
            specs.append(
                {
                    "mode": "gpu_v3_custom_topq",
                    "method_variant": "random_support_lift",
                    "top_q": tq,
                    "eig_method": "random_support_lift",
                    "method_cfg": {
                        "s": int(EXTRA_SUPPORT_LIFT_S),
                        "q_each": int(EXTRA_SUPPORT_LIFT_Q_EACH),
                        "n_sketches": int(EXTRA_SUPPORT_LIFT_N_SKETCHES),
                        "r_full": int(r_full_eff),
                        "block_cols": int(EXTRA_SUPPORT_LIFT_BLOCK_COLS),
                    },
                }
            )

        if _eig_enabled("extra_chebyshev_filtered_subspace"):
            if EXTRA_CHEBY_LAMBDA_LOW is None or EXTRA_CHEBY_LAMBDA_CUT is None:
                raise ValueError("extra_chebyshev_filtered_subspace 已启用，但 EXTRA_CHEBY_LAMBDA_LOW / CUT 仍为 None")
            specs.append(
                {
                    "mode": "gpu_v3_custom_topq",
                    "method_variant": "chebyshev_filtered_subspace",
                    "top_q": tq,
                    "eig_method": "chebyshev_filtered_subspace",
                    "method_cfg": {
                        "degree": int(EXTRA_CHEBY_DEGREE),
                        "lambda_low": float(EXTRA_CHEBY_LAMBDA_LOW),
                        "lambda_cut": float(EXTRA_CHEBY_LAMBDA_CUT),
                        "oversample": int(EXTRA_CHEBY_OVERSAMPLE),
                        "block_cols": int(EXTRA_CHEBY_BLOCK_COLS),
                    },
                }
            )
    return specs


_nystrom_variants_active = (
    _active_variants(EIGENPRO_NYSTROM_MAIN_VARIANTS)
    + _active_variants(EIGENPRO_NYSTROM_LEVELS_VARIANTS)
    + _active_variants(EIGENPRO_NYSTROM_ABLATION_VARIANTS)
)

MODE_SPECS = (
    ([{"mode": "gpu_v1_topq0", "method_variant": "baseline_v1_topq0", "top_q": 0}] if _eig_enabled("baseline_v1_topq0") else [])
    + ([{"mode": "gpu_v3_topq", "method_variant": "baseline_v3_topq", "top_q": int(q)} for q in V3_TOPQ_LIST] if _eig_enabled("baseline_v3_topq") else [])
    + _build_nystrom_mode_specs(NYSTROM_TOPQ_LIST, _nystrom_variants_active)
    + _build_extra_eig_mode_specs(EXTRA_TOPQ_LIST)
)

print("EIG_METHOD_TOGGLES:", EIG_METHOD_TOGGLES)
print("MODE_SPECS:", MODE_SPECS)


In [ ]:
# ---- Dataset helpers / generic utilities ----

def _sync_gpu():
    if cp is not None:
        cp.cuda.Stream.null.synchronize()


def _clear_state(clear_pool: bool = False):
    gc.collect()
    if cp is not None:
        try:
            _sync_gpu()
        except Exception:
            pass
        if clear_pool:
            cp.get_default_memory_pool().free_all_blocks()
            cp.get_default_pinned_memory_pool().free_all_blocks()
            _sync_gpu()


def _gpu_mem_used_gb() -> float:
    if cp is None:
        return float("nan")
    try:
        free_b, total_b = cp.cuda.runtime.memGetInfo()
        return float((total_b - free_b) / (1024 ** 3))
    except Exception:
        return float("nan")


def _device_name() -> str:
    if cp is None:
        return "cpu_only"
    try:
        return cp.cuda.runtime.getDeviceProperties(0)["name"].decode("utf-8")
    except Exception:
        return "unknown_gpu"


def _make_kernel(kernel_cfg: dict, dim: int):
    fam = str(kernel_cfg.get("family", "matern")).strip().lower()
    lengthscale = float(kernel_cfg["lengthscale"])
    variance = float(kernel_cfg.get("variance", 1.0))
    if fam in ("matern", "mat", "mat32", "mat52"):
        nu = float(kernel_cfg.get("nu", 1.5))
        return make_matern(lengthscale=lengthscale, nu=nu, dim=int(dim), variance=variance)
    if fam in ("se", "squared_exponential", "squared-exponential", "rbf", "gaussian"):
        return make_squared_exponential(lengthscale=lengthscale, dim=int(dim), variance=variance)
    raise ValueError(f"Unsupported kernel family: {fam}")


def _load_sidecar_metadata(npz_path: Path) -> dict:
    json_path = npz_path.with_suffix(".json")
    if not json_path.exists():
        return {}
    try:
        return json.loads(json_path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def _sanitize_dataset_name(name: str) -> str:
    base = Path(str(name)).stem
    return re.sub(r"[^0-9a-zA-Z_\-]+", "_", base).strip("_")


def _dataset_output_paths(dataset_name: str) -> dict:
    dataset_tag = _sanitize_dataset_name(dataset_name)
    dataset_dir = DATASET_OUTPUTS_DIR / dataset_tag
    dataset_dir.mkdir(parents=True, exist_ok=True)
    return {
        "dataset_tag": dataset_tag,
        "dataset_dir": dataset_dir,
        "raw_csv": dataset_dir / "raw_runs.csv",
        "summary_csv": dataset_dir / "summary.csv",
        "env_json": dataset_dir / "env_info.json",
        "slq_dir": dataset_dir / "slq_diagnostics",
    }


def build_dataset_specs() -> list[dict]:
    # Accept either bare stems or filenames in DATASET_SELECTION.
    discovered = dict(DISCOVERED_DATASET_MAP)
    missing = []
    selected_keys = []
    for name in DATASET_SELECTION:
        key = Path(str(name)).stem
        if key not in discovered:
            missing.append(str(name))
        selected_keys.append(key)
    if missing:
        raise FileNotFoundError(f"Selected processed datasets not found: {missing}")

    specs = []
    for key in selected_keys:
        path = discovered[key]
        specs.append(
            {
                "name": key,
                "path": path,
                "metadata_path": path.with_suffix(".json"),
            }
        )
    return specs


def load_dataset(spec: dict) -> dict:
    path = Path(spec["path"])
    meta = _load_sidecar_metadata(path)
    loaded = np.load(path)
    required = ("x_train", "x_test", "y_train", "y_test")
    missing = [k for k in required if k not in loaded.files]
    if missing:
        raise ValueError(f"Processed dataset {path.name} missing arrays: {missing}")

    x_train = np.asarray(loaded["x_train"], dtype=np.float64)
    x_test = np.asarray(loaded["x_test"], dtype=np.float64)
    y_train = np.asarray(loaded["y_train"], dtype=np.float64).reshape(-1)
    y_test = np.asarray(loaded["y_test"], dtype=np.float64).reshape(-1)
    if x_train.ndim != 2 or x_test.ndim != 2:
        raise ValueError(f"Processed dataset {path.name} must store 2D x_train/x_test")
    if x_train.shape[1] != x_test.shape[1]:
        raise ValueError(f"Processed dataset {path.name} has mismatched train/test feature dims")

    shapes = meta.get("shapes", {}) if isinstance(meta, dict) else {}
    n_total = shapes.get("n_clean", shapes.get("n_total", int(x_train.shape[0] + x_test.shape[0])))

    return {
        "name": spec["name"],
        "path": str(path),
        "metadata_path": str(spec["metadata_path"]),
        "dim": int(x_train.shape[1]),
        "n_total": int(n_total),
        "n_train": int(x_train.shape[0]),
        "n_test": int(x_test.shape[0]),
        "x_train": x_train,
        "y_train": y_train,
        "x_test": x_test,
        "y_test": y_test,
        "metadata": meta,
        "available_arrays": sorted(list(loaded.files)),
    }


def _resolve_precompute_methods(dataset_payload: dict) -> list[str]:
    n_total = int(dataset_payload.get("n_total", 0))
    threshold = PRECOMPUTE_C1_MIN_N_TOTAL
    resolved = []
    for pcm in PRECOMPUTE_METHODS:
        pcm_str = str(pcm).strip()
        if pcm_str.lower() == "c1" and threshold is not None and n_total < int(threshold):
            continue
        resolved.append(pcm_str)
    if not resolved:
        raise ValueError(f"No precompute methods enabled for dataset={dataset_payload.get('name', '<unknown>')}")
    return resolved


def _collect_env_info(dataset_specs: list[dict]) -> dict:
    info = {
        "timestamp": datetime.now().isoformat(),
        "python": sys.version,
        "platform": sys.platform,
        "device_name": _device_name(),
        "raw_data_dir": str(RAW_DATA_DIR),
        "processed_data_dir": str(PROCESSED_DATA_DIR),
        "datasets": [
            {
                "name": s["name"],
                "path": str(s["path"]),
                "metadata_path": str(s["metadata_path"]),
            }
            for s in dataset_specs
        ],
        "dataset_selection": list(DATASET_SELECTION),
        "kernel_specs": KERNEL_SPECS,
        "eps_list": list(EPS_LIST),
        "mode_specs": MODE_SPECS,
        "precompute_methods": PRECOMPUTE_METHODS,
        "precompute_c1_min_n_total": PRECOMPUTE_C1_MIN_N_TOTAL,
        "reg_lambda": REG_LAMBDA,
        "solve_tol": SOLVE_TOL,
        "gpu_maxiter": GPU_MAXITER,
        "gpu_nufft": GPU_NUFFT,
        "l2_scaled": L2_SCALED,
        "eigenpro_nystrom": {
            "precond_kind": EIGENPRO_NYSTROM_PRECOND_KIND,
            "refine_mode": EIGENPRO_NYSTROM_REFINE_MODE,
            "refine_iters": EIGENPRO_NYSTROM_REFINE_ITERS,
            "topq_list": list(NYSTROM_TOPQ_LIST),
            "main_variants": list(EIGENPRO_NYSTROM_MAIN_VARIANTS),
            "include_ablation": bool(EIGENPRO_NYSTROM_INCLUDE_ABLATION),
            "ablation_variants": list(EIGENPRO_NYSTROM_ABLATION_VARIANTS),
            "surrogate_size": EIGENPRO_NYSTROM_SURROGATE_SIZE,
            "lowfreq_ratio": EIGENPRO_NYSTROM_LOWFREQ_RATIO,
            "oversample": EIGENPRO_NYSTROM_OVERSAMPLE,
            "ritz_refine": EIGENPRO_NYSTROM_RITZ_REFINE,
            "seed": EIGENPRO_NYSTROM_SEED,
            "block_rows": EIGENPRO_NYSTROM_BLOCK_ROWS,
            "ritz_block_cols": EIGENPRO_NYSTROM_RITZ_BLOCK_COLS,
            "lift": EIGENPRO_NYSTROM_LIFT,
        },
        "enable_slq": bool(ENABLE_SLQ),
    }
    try:
        import cupy
        info["cupy"] = cupy.__version__
    except Exception:
        info["cupy"] = "unavailable"
    try:
        import cufinufft
        info["cufinufft"] = getattr(cufinufft, "__version__", "unknown")
    except Exception:
        info["cufinufft"] = "unavailable"
    return info


def _extract_common_metrics(diag: dict) -> dict:
    return {
        "cg_iters": int(diag.get("cg_iters", -1)),
        "cg_relres": float(diag.get("cg_relres", np.nan)),
        "n_matvec": int(diag.get("n_matvec", 0)),
        "t_matvec_total": float(diag.get("t_matvec_total", np.nan)),
        "n_precond": int(diag.get("n_precond", 0)),
        "t_precond_total": float(diag.get("t_precond_total", np.nan)),
        "time_eigenspace": float(diag.get("time_eigenspace", 0.0)),
        "time_precond_build": float(diag.get("time_precond_build", 0.0)),
        "time_solve": float(diag.get("time_solve", np.nan)),
        "time_predict": float(diag.get("time_predict", np.nan)),
        "nufft_stage": str(diag.get("nufft_stage", "")),
        "device_name": str(diag.get("device_name", "")),
        "eig_nystrom_kernel_s": float(diag.get("eig_nystrom_kernel_s", np.nan)),
        "surrogate_tag": str(diag.get("surrogate_tag", "")),
        "precond_kind": str(diag.get("precond_kind", "")),
        "surrogate_refine_mode": str(diag.get("surrogate_refine_mode", "")),
        "surrogate_refine_mode_requested": str(diag.get("surrogate_refine_mode_requested", "")),
        "coord_nystrom_gamma": float(diag.get("coord_nystrom_gamma", np.nan)),
        "lambda1_coord_nystrom": float(diag.get("lambda1_coord_nystrom", np.nan)),
    }


def _label_case(mode: str, top_q: int, precompute_method: str, eps: float, kernel_name: str) -> str:
    return f"{kernel_name} | eps={eps:g} | {mode} | q={int(top_q)} | pcm={str(precompute_method).lower()}"

In [ ]:
# ---- GPU benchmark patch / runners ----
_binned_pc_mod = importlib.reload(_binned_pc_mod)
build_binned_efgp_system = _binned_pc_mod.build_binned_efgp_system

_BENCHMARK_PC_METHOD_ACTIVE = None
_LAST_PC_PATCH_EXTRA = {}


def _grid_mtot_and_M(x_train: np.ndarray, kernel, eps: float, *, l2scaled: bool) -> tuple[int, int]:
    x = np.asarray(x_train, dtype=np.float64)
    x_min = np.min(x, axis=0)
    x_max = np.max(x, axis=0)
    L = float(np.max(x_max - x_min))
    grid = choose_grid_params(kernel, float(eps), L, l2scaled=bool(l2scaled))
    mtot = int(grid.mtot)
    dim = int(getattr(kernel, "dim", x.shape[1]))
    M = int(mtot ** dim)
    return mtot, M


def _resolve_nystrom_precond_kind(precond_kind_requested: str, M: int) -> tuple[str, str]:
    req = str(precond_kind_requested).strip().lower()
    eff = req
    thr = EIGENPRO_COORD_NYSTROM_MIN_M
    if req in ("coordinate_nystrom", "coord_nystrom") and thr is not None:
        if int(M) <= int(thr):
            eff = "full_eigenpro"
    return req, eff


def make_eigenpro_nystrom_eigenspace_config(
    top_q: int,
    *,
    precond_kind=None,
    refine_mode=None,
    refine_iters=None,
    coord_gamma=None,
    extra_method_cfg=None,
) -> EigenspaceConfig:
    tq = int(top_q)
    s_nys = int(EIGENPRO_NYSTROM_SURROGATE_SIZE) if EIGENPRO_NYSTROM_SURROGATE_SIZE is not None else 10 * (tq + 1)
    br = EIGENPRO_NYSTROM_BLOCK_ROWS
    pk = str(precond_kind if precond_kind is not None else EIGENPRO_NYSTROM_PRECOND_KIND).lower()
    rm = refine_mode if refine_mode is not None else EIGENPRO_NYSTROM_REFINE_MODE
    ri = int(refine_iters if refine_iters is not None else EIGENPRO_NYSTROM_REFINE_ITERS)
    gg = float(coord_gamma if coord_gamma is not None else EIGENPRO_COORD_NYSTROM_GAMMA)
    mcfg = {"precond_kind": pk, "coord_nystrom_gamma": gg}
    if extra_method_cfg:
        mcfg.update(dict(extra_method_cfg))
    return EigenspaceConfig(
        q_max=tq,
        block_size=max(s_nys, tq + 1),
        n_iter=0,
        eig_method="eigenpro_nystrom",
        method_cfg=mcfg,
        surrogate_size=s_nys,
        surrogate_oversample=int(EIGENPRO_NYSTROM_OVERSAMPLE),
        surrogate_lowfreq_ratio=float(EIGENPRO_NYSTROM_LOWFREQ_RATIO),
        surrogate_ritz_refine=bool(EIGENPRO_NYSTROM_RITZ_REFINE),
        surrogate_seed=int(EIGENPRO_NYSTROM_SEED),
        surrogate_block_rows=None if br is None else int(br),
        surrogate_ritz_block_cols=int(EIGENPRO_NYSTROM_RITZ_BLOCK_COLS),
        surrogate_lift=bool(EIGENPRO_NYSTROM_LIFT),
        surrogate_refine_mode=str(rm) if rm is not None else "auto",
        surrogate_refine_iters=int(max(0, ri)),
    )


def _install_gpu_rhs_benchmark_patch() -> None:
    global _GPU_PC_ORIGINAL_FN

    import importlib as _il

    _il.reload(_gpu_v1_ops_bm)
    _GPU_PC_ORIGINAL_FN = _gpu_v1_ops_bm.gpu_precompute_v1

    def _wrapped(
        backend,
        kernel,
        eps,
        nufft_tol,
        data_ctx,
        op_ctx=None,
        *,
        l2scaled=False,
        force=False,
        chunk_size=None,
    ):
        global _LAST_PC_PATCH_EXTRA
        pcm = (_BENCHMARK_PC_METHOD_ACTIVE or "gpu_exact").strip().lower()

        if pcm not in ("c0", "c1", "c2"):
            t_nu0 = time.perf_counter()
            ctx = _GPU_PC_ORIGINAL_FN(
                backend,
                kernel,
                eps,
                nufft_tol,
                data_ctx,
                op_ctx,
                l2scaled=l2scaled,
                force=force,
                chunk_size=chunk_size,
            )
            t_nufft = float(time.perf_counter() - t_nu0)
            _LAST_PC_PATCH_EXTRA = {
                "t_original_exact_gpu_precompute_v1_s": t_nufft,
                "time_precompute_NUFFT": t_nufft,
                "time_precompute_binned": float(np.nan),
            }
            return ctx

        xp = backend.xp
        ctx = data_ctx
        X = xp.asarray(ctx.x_gpu, dtype=xp.float64)
        y = xp.asarray(ctx.y_gpu, dtype=xp.float64).reshape(-1)
        n = int(X.shape[0])
        dim = int(kernel.dim)

        x_min = xp.min(X, axis=0)
        x_max = xp.max(X, axis=0)
        L = float(xp.max(x_max - x_min))
        x_center_gpu = (x_min + x_max) / 2.0
        grid = choose_grid_params(kernel, eps, L, l2scaled=l2scaled)
        mtot = int(grid.mtot)
        hm = (mtot - 1) // 2
        if mtot != 2 * hm + 1:
            raise RuntimeError(f"unexpected mtot={mtot}")

        weights_np = np.ascontiguousarray(basis_weights(kernel, grid.xis, grid.h).reshape(-1))
        w_gpu = xp.asarray(weights_np, dtype=xp.float64)
        weights_flat = w_gpu.reshape(-1)
        weights_nd = weights_flat.reshape((mtot,) * dim)
        x_center_np = np.asarray(_device_array_to_numpy(x_center_gpu, np.float64)).reshape(-1)

        t_bin0 = time.perf_counter()
        v_tilde, b_tilde, diag_bin = build_binned_efgp_system(
            X,
            y,
            n,
            dim,
            float(grid.h),
            hm,
            weights_np,
            order=pcm.upper(),
            quality=str(BINNED_QUALITY),
            r=int(BINNED_R_USER) if BINNED_R_USER is not None else None,
            use_sparse_bins=bool(BINNED_USE_SPARSE_BINS),
            use_gpu_dense_bins=bool(BINNED_USE_GPU_DENSE_BINS),
            return_bin_stats=False,
            x_center=x_center_np,
            backend=backend,
            nufft_tol=float(nufft_tol),
            gpu_timing=True,
            input_on_gpu=True,
            assume_normalized=True,
            skip_cpu_validation=True,
            allow_exact_nufft_fallback=bool(BINNED_ALLOW_EXACT_NUFFT_FALLBACK),
            nufft_allow_cpu_fallback=bool(BINNED_NUFFT_ALLOW_CPU_FALLBACK),
        )

        ms_xtx = 2 * int(mtot) - 1
        exp_modes = int(ms_xtx) ** int(dim)
        vt_flat = xp.asarray(v_tilde).reshape(-1)
        if int(vt_flat.size) != exp_modes:
            raise RuntimeError(f"binned v_tilde size {vt_flat.size} != expected {exp_modes}")
        xtxcol_gpu = xp.ascontiguousarray(vt_flat.reshape((ms_xtx,) * int(dim)))
        Gf_gpu = xp.ascontiguousarray(backend.fft.fftn(xtxcol_gpu))

        exp_sz = int(mtot ** dim)
        if cp is None or not isinstance(b_tilde, cp.ndarray):
            raise RuntimeError("GPU-only benchmark expects build_binned_efgp_system to return GPU b_tilde.")
        rhs_gpu = b_tilde.reshape(-1).astype(xp.complex128, copy=False)
        if int(rhs_gpu.size) != exp_sz:
            raise RuntimeError(f"b_tilde size {rhs_gpu.size} != expected {exp_sz}")

        ctx.weights_gpu_nd = weights_nd
        ctx.weights_gpu_flat = weights_flat
        ctx.weights_np_flat = np.ascontiguousarray(weights_np.reshape(-1))
        ctx.rhs_gpu = rhs_gpu
        ctx.xtxcol_gpu = xtxcol_gpu
        ctx.gf_gpu = Gf_gpu
        ctx.x_center_gpu = x_center_gpu

        _sync_gpu()
        t_bin_wall = float(time.perf_counter() - t_bin0)
        bd = diag_bin.get("binned_precompute_breakdown_s", None) or {}
        _LAST_PC_PATCH_EXTRA = {
            "t_original_exact_gpu_precompute_v1_s": float(np.nan),
            "time_precompute_NUFFT": float(np.nan),
            "time_precompute_binned": t_bin_wall,
            "precompute_benchmark_note": "binned pcm path: build_binned_efgp_system provides XtXcol + rhs; exact gpu_precompute_v1 is skipped.",
            "binned_theta_actual": float(diag_bin.get("theta_actual", np.nan)),
            "binned_G": int(diag_bin.get("G", -1)),
            "binned_num_occupied_bins": int(diag_bin.get("num_occupied_bins", -1)),
            "effective_work_ratio": float(diag_bin.get("effective_work_ratio", np.nan)),
            "binned_used_exact_dense_point_nufft": float(bool(diag_bin.get("used_exact_dense_point_nufft", False))),
            "binned_order_bins_final": str(diag_bin.get("order_bins_final", "")),
            "binned_allow_exact_nufft_fallback": float(bool(diag_bin.get("allow_exact_nufft_fallback", False))),
        }
        for k, v in bd.items():
            try:
                _LAST_PC_PATCH_EXTRA[str(k)] = float(v)
            except (TypeError, ValueError):
                _LAST_PC_PATCH_EXTRA[str(k)] = float(np.nan)

        ctx.meta.update(
            {
                "mtot": mtot,
                "dim": dim,
                "h": float(grid.h),
                "weight_shape": tuple(int(s) for s in ctx.weights_gpu_nd.shape),
                "gf_shape": tuple(int(s) for s in ctx.gf_gpu.shape),
                "rhs_shape": tuple(int(s) for s in ctx.rhs_gpu.shape),
                "nufft_tol": float(nufft_tol),
                "nufft_stage": f"binned_{pcm}",
                "chunk_size": None,
                "gf_absmax": float(xp.max(xp.abs(ctx.gf_gpu))),
                "debug_finite_checks": bool(ctx.meta.get("debug_finite_checks", False)),
                "rhs_variant": pcm.upper(),
            }
        )
        return ctx

    _gpu_v1_ops_bm.gpu_precompute_v1 = _wrapped
    _gpu_versions_bm.gpu_precompute_v1 = _wrapped
    if hasattr(_gpu_pkg_bm, "gpu_precompute_v1"):
        _gpu_pkg_bm.gpu_precompute_v1 = _wrapped
    _gpu_v1_ops_bm._benchmark_rhs_patch_installed = True


_install_gpu_rhs_benchmark_patch()


def _gpu_scalar(x) -> float:
    arr = np.asarray(_device_array_to_numpy(x)).reshape(-1)
    if arr.size == 0:
        return float("nan")
    return float(arr[0])


def _gpu_regression_metrics(backend, yhat_gpu, y_true_np: np.ndarray) -> tuple[float, float, float]:
    xp = backend.xp
    y_true_gpu = xp.asarray(np.asarray(y_true_np, dtype=np.float64))
    yhat_gpu = xp.asarray(yhat_gpu, dtype=xp.float64).reshape(-1)
    y_true_gpu = y_true_gpu.reshape(-1)
    resid_gpu = yhat_gpu - y_true_gpu

    rmse = _gpu_scalar(xp.sqrt(xp.mean(resid_gpu * resid_gpu)))
    mae = _gpu_scalar(xp.mean(xp.abs(resid_gpu)))
    ss_res = _gpu_scalar(xp.sum(resid_gpu * resid_gpu))

    centered_true = y_true_gpu - xp.mean(y_true_gpu)
    ss_tot = _gpu_scalar(xp.sum(centered_true * centered_true))
    r2 = float("nan") if ss_tot <= 0.0 else float(1.0 - (ss_res / ss_tot))
    return rmse, mae, r2


# ---- Eigenvalue diagnostics (coord_nystrom vs default_v3 vs cupy_eigsh(A)) ----
_LAMBDA1_A_EIGSH_CACHE = {}
_LAMBDA_DEFAULT_V3_CACHE = {}


def _fmt_sci_sig3(x) -> str:
    try:
        v = float(x)
    except Exception:
        return "nan"
    if not np.isfinite(v):
        return "nan"
    return f"{v:.3e}"


def _should_run_lambda1_diag() -> bool:
    try:
        return any(str(s.get("mode", "")) == "gpu_v3_topq_eigenpro_nystrom" for s in MODE_SPECS)
    except Exception:
        return False


def _build_apply_A_block_for_diag(backend, data_ctx, reg_lambda: float):
    from efgp_eigenpro_py.gpu.contexts import GPUOperatorContext
    from efgp_eigenpro_py.gpu.v1_ops import apply_A_v1

    xp = backend.xp
    op_ctx = GPUOperatorContext()

    def _apply_A_block(V):
        V = xp.asarray(V, dtype=xp.complex128)
        if V.ndim == 1:
            V = V.reshape(-1, 1)
        out = xp.empty_like(V)
        for i in range(int(V.shape[1])):
            apply_A_v1(backend, data_ctx, V[:, i], float(reg_lambda), op_ctx, out=out[:, i])
        return out

    return _apply_A_block


def _eigvals_A_via_cupy_eigsh(backend, data_ctx, reg_lambda: float, top_q: int) -> list[float]:
    # Estimate top-q eigenvalues of A using cupy_eigsh on the GPU linear operator.
    from efgp_eigenpro_py.gpu.v3_eigenspace import estimate_top_eigenspace_v3

    tq = int(top_q)
    eig_cfg = EigenspaceConfig(
        q_max=tq,
        block_size=max(64, tq + 32),
        n_iter=1,
        eig_method="cupy_eigsh",
        method_cfg={
            "which": "LA",
            "tol": 1e-6,
            "maxiter": 300,
            "ncv": max(96, 2 * (tq + 1) + 32),
            "warm_start_strategy": "power1",
        },
    )
    vals, _vecs, _diag = estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=_build_apply_A_block_for_diag(backend, data_ctx, reg_lambda),
        size=int(data_ctx.rhs_gpu.size),
        cfg=eig_cfg,
    )
    return [float(v) for v in np.asarray(_device_array_to_numpy(vals)).reshape(-1)[:tq]]


def _eigvals_A_via_default_v3(backend, data_ctx, reg_lambda: float, top_q: int) -> list[float]:
    # Estimate top-q eigenvalues using the notebook's default v3 eigenspace solver.
    from efgp_eigenpro_py.gpu.v3_eigenspace import estimate_top_eigenspace_v3

    tq = int(top_q)
    eig_cfg = EigenspaceConfig(
        q_max=tq,
        block_size=int(tq + V3_OVERSAMPLE),
        n_iter=int(V3_N_ITER),
    )
    vals, _vecs, _diag = estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=_build_apply_A_block_for_diag(backend, data_ctx, reg_lambda),
        size=int(data_ctx.rhs_gpu.size),
        cfg=eig_cfg,
    )
    return [float(v) for v in np.asarray(_device_array_to_numpy(vals)).reshape(-1)[:tq]]


def _run_case_once(
    dataset_payload: dict,
    kernel_cfg: dict,
    eps: float,
    mode_spec: dict,
    precompute_method: str,
    repeat_idx: int,
    warmup_only: bool = False,
) -> dict:
    global _BENCHMARK_PC_METHOD_ACTIVE

    dataset_name = str(dataset_payload["name"])
    dataset_paths = _dataset_output_paths(dataset_name)
    x_train = np.asarray(dataset_payload["x_train"], dtype=np.float64)
    y_train = np.asarray(dataset_payload["y_train"], dtype=np.float64)
    x_test = np.asarray(dataset_payload["x_test"], dtype=np.float64)
    y_test = np.asarray(dataset_payload["y_test"], dtype=np.float64)
    dim = int(dataset_payload["dim"])
    mode = str(mode_spec["mode"])
    mode_requested = mode
    mode_effective = mode
    top_q = int(mode_spec.get("top_q", 0))
    pcm_lc = str(precompute_method).strip().lower()

    nystrom_gate_threshold_M = EIGENPRO_COORD_NYSTROM_MIN_M
    nystrom_gate_triggered = False
    nystrom_grid_mtot = np.nan
    nystrom_grid_M = np.nan
    nystrom_variant = str(mode_spec.get("nystrom_variant", "default"))
    method_variant = str(mode_spec.get("method_variant", nystrom_variant if mode == "gpu_v3_topq_eigenpro_nystrom" else mode))
    eig_method_requested = str(mode_spec.get("eig_method", "subspace_iter" if mode in ("gpu_v3_topq", "gpu_v3_custom_topq") else ("eigenpro_nystrom" if mode == "gpu_v3_topq_eigenpro_nystrom" else "")))
    eig_method_effective = eig_method_requested
    nystrom_precond_kind_requested = str(mode_spec.get("nystrom_precond_kind", EIGENPRO_NYSTROM_PRECOND_KIND)).strip().lower()
    nystrom_precond_kind_effective = nystrom_precond_kind_requested
    nystrom_refine_mode_requested = mode_spec.get("nystrom_refine_mode", EIGENPRO_NYSTROM_REFINE_MODE)
    nystrom_refine_mode_effective = nystrom_refine_mode_requested
    nystrom_refine_iters_raw = mode_spec.get("nystrom_refine_iters", EIGENPRO_NYSTROM_REFINE_ITERS)
    if nystrom_refine_iters_raw is None:
        nystrom_refine_iters_raw = EIGENPRO_NYSTROM_REFINE_ITERS
    nystrom_refine_iters_requested = int(nystrom_refine_iters_raw)

    kernel = _make_kernel(kernel_cfg, dim)
    solver = EFGPSolver(
        kernel=kernel,
        reg_lambda=REG_LAMBDA,
        eps=float(eps),
        nufft_tol=1e-10,
        l2scaled=L2_SCALED,
    )
    cfg = GPURunConfig(
        reg_lambda=REG_LAMBDA,
        tol=SOLVE_TOL,
        maxiter=GPU_MAXITER,
        chunk_size=None,
        debug_finite_checks=DEBUG_FINITE_CHECKS,
        backend=BackendConfig(nufft=GPU_NUFFT),
    )

    try:
        _BENCHMARK_PC_METHOD_ACTIVE = pcm_lc
        _sync_gpu()
        mem_before = _gpu_mem_used_gb()
        t0 = time.perf_counter()

        if mode == "gpu_v1_topq0":
            out = run_v1_pure_efgp(solver, x_train, y_train, cfg)
        elif mode == "gpu_v3_topq":
            if top_q <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq")
            eig_cfg = EigenspaceConfig(
                q_max=int(top_q),
                block_size=int(top_q + V3_OVERSAMPLE),
                n_iter=int(V3_N_ITER),
            )
            out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        elif mode == "gpu_v3_custom_topq":
            if top_q <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_custom_topq")
            eig_method = str(mode_spec.get("eig_method", "subspace_iter"))
            method_cfg = dict(mode_spec.get("method_cfg", {}) or {})
            eig_cfg = EigenspaceConfig(
                q_max=int(top_q),
                block_size=int(top_q + V3_OVERSAMPLE),
                n_iter=int(V3_N_ITER),
                eig_method=eig_method,
                method_cfg=method_cfg,
            )
            out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        elif mode == "gpu_v3_topq_eigenpro_nystrom":
            if top_q <= 0:
                raise ValueError("top_q must be > 0 for gpu_v3_topq_eigenpro_nystrom")

            mtot_case, M_case = _grid_mtot_and_M(x_train, kernel, float(eps), l2scaled=L2_SCALED)
            nystrom_grid_mtot = int(mtot_case)
            nystrom_grid_M = int(M_case)

            # Gate: when M is small, Nyström-based eigenspace may be unstable; fall back to default gpu_v3_topq.
            if nystrom_gate_threshold_M is not None and int(M_case) <= int(nystrom_gate_threshold_M):
                nystrom_gate_triggered = True
                mode_effective = "gpu_v3_topq"
                mode = mode_effective
                nystrom_refine_mode_effective = "fallback_gpu_v3_topq"
                print(
                    "[WARN] Nyström eigenspace may be unstable at small grid size; "
                    f"switching to gpu_v3_topq. dataset={dataset_name} kernel={kernel_cfg['name']} eps={eps:g} q={top_q} "
                    f"mtot={mtot_case} M={M_case} threshold_M={nystrom_gate_threshold_M}"
                )
                eig_cfg = EigenspaceConfig(
                    q_max=int(top_q),
                    block_size=int(top_q + V3_OVERSAMPLE),
                    n_iter=int(V3_N_ITER),
                )
                out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
            else:
                pk_req, pk_eff = _resolve_nystrom_precond_kind(nystrom_precond_kind_requested, int(M_case))
                nystrom_precond_kind_requested = str(pk_req)
                nystrom_precond_kind_effective = str(pk_eff)
                nystrom_refine_mode_effective = nystrom_refine_mode_requested
                coord_gamma = float(mode_spec.get("coord_gamma", EIGENPRO_COORD_NYSTROM_GAMMA))
                nystrom_method_cfg = dict(mode_spec.get("nystrom_method_cfg", {}) or {})
                eig_cfg = make_eigenpro_nystrom_eigenspace_config(
                    int(top_q),
                    precond_kind=pk_eff,
                    refine_mode=nystrom_refine_mode_requested,
                    refine_iters=nystrom_refine_iters_requested,
                    coord_gamma=coord_gamma,
                    extra_method_cfg=nystrom_method_cfg,
                )
                out = run_v3_full_gpu_eigenspace(solver, x_train, y_train, cfg, eig_cfg)
        else:
            raise ValueError(f"Unsupported mode: {mode}")

        _sync_gpu()
        t1 = time.perf_counter()
        mem_after = _gpu_mem_used_gb()

        if warmup_only:
            try:
                del out
            except Exception:
                pass
            return {"status": "warmup_done"}

        if out.backend is None or out.data_ctx is None:
            raise RuntimeError("GPU run output is missing backend/data_ctx for GPU prediction.")

        yhat_test_gpu = predict_v1(out.backend, out.data_ctx, x_test, out.beta_gpu)
        yhat_train_gpu = predict_v1(out.backend, out.data_ctx, x_train, out.beta_gpu)
        diag = out.diagnostics
        eig_method_effective = str(diag.get("method", eig_method_requested))
        if mode_requested == "gpu_v3_topq_eigenpro_nystrom" and (not bool(nystrom_gate_triggered)):
            nystrom_refine_mode_effective = str(
                diag.get("surrogate_refine_mode", diag.get("surrogate_refine_mode_requested", nystrom_refine_mode_requested))
            )
        rmse_test, mae_test, r2_test = _gpu_regression_metrics(out.backend, yhat_test_gpu, y_test)
        rmse_train, _, _ = _gpu_regression_metrics(out.backend, yhat_train_gpu, y_train)

        # Record grid size stats (M=mtot^dim) and Nyström gating decision.
        mtot_meta = int(out.data_ctx.meta.get("mtot", -1)) if out.data_ctx is not None else -1
        M_meta = int(mtot_meta ** int(dim)) if mtot_meta > 0 else -1

        row = {
            "run_id": f"{RUN_TAG}_{dataset_name}_{kernel_cfg['name']}_{mode}_pcm{pcm_lc}_q{top_q}_eps{eps:g}_rep{repeat_idx}",
            "timestamp": datetime.now().isoformat(),
            "dataset": dataset_name,
            "dataset_tag": str(dataset_paths["dataset_tag"]),
            "dataset_path": str(dataset_payload["path"]),
            "dataset_output_dir": str(dataset_paths["dataset_dir"]),
            "dim": int(dim),
            "n_total": int(dataset_payload["n_total"]),
            "n_train": int(x_train.shape[0]),
            "n_test": int(x_test.shape[0]),
            "kernel_name": str(kernel_cfg["name"]),
            "kernel_family": str(kernel_cfg.get("family", "matern")),
            "kernel_lengthscale": float(kernel_cfg["lengthscale"]),
            "kernel_nu": float(kernel_cfg.get("nu", np.nan)) if kernel_cfg.get("nu", None) is not None else np.nan,
            "kernel_variance": float(kernel_cfg.get("variance", 1.0)),
            "eps": float(eps),
            "mode": mode,
            "mode_requested": mode_requested,
            "mode_effective": mode_effective,
            "method_variant": method_variant,
            "eig_method_requested": eig_method_requested,
            "eig_method_effective": eig_method_effective,
            "top_q": int(top_q),
            "precompute_method": pcm_lc,
            "grid_mtot": int(mtot_meta),
            "grid_M": int(M_meta),
            "nystrom_grid_mtot": float(nystrom_grid_mtot),
            "nystrom_grid_M": float(nystrom_grid_M),
            "nystrom_gate_threshold_M": float(nystrom_gate_threshold_M) if nystrom_gate_threshold_M is not None else np.nan,
            "nystrom_gate_triggered": float(bool(nystrom_gate_triggered)),
            "nystrom_variant": str(nystrom_variant),
            "eigenpro_nystrom_precond_kind_requested": str(nystrom_precond_kind_requested),
            "eigenpro_nystrom_precond_kind_effective": str(nystrom_precond_kind_effective),
            "eigenpro_nystrom_refine_mode_requested": str(nystrom_refine_mode_requested),
            "eigenpro_nystrom_refine_mode_effective": str(nystrom_refine_mode_effective),
            "eigenpro_nystrom_refine_iters_requested": int(nystrom_refine_iters_requested),
            "eigenpro_coord_nystrom_min_M": int(EIGENPRO_COORD_NYSTROM_MIN_M) if EIGENPRO_COORD_NYSTROM_MIN_M is not None else np.nan,
            "reg_lambda": float(REG_LAMBDA),
            "cg_tol": float(SOLVE_TOL),
            "wall_s_outer_s": float(t1 - t0),
            "rmse_train": rmse_train,
            "rmse_test": rmse_test,
            "mae_test": mae_test,
            "r2_test": r2_test,
            "peak_mem_gb": float(np.nanmax([mem_before, mem_after])),
            "status": "ok",
            "error": "",
            "repeat_idx": int(repeat_idx),
            "repeat_count": int(REPEATS),
        }
        row.update(_extract_common_metrics(diag))
        row.update(dict(_LAST_PC_PATCH_EXTRA))

        # Eigenvalue diagnostics: compare coordinate_nystrom theta_i(W) vs lambda_i(A) from cupy_eigsh.
        if (
            _should_run_lambda1_diag()
            and mode_requested == "gpu_v3_topq_eigenpro_nystrom"
            and str(row.get("precond_kind", "")) == "coordinate_nystrom"
            and (not bool(nystrom_gate_triggered))
            and cp is not None
        ):
            try:
                theta_list = list(diag.get("theta_coord_topq", []) or [])
                eps_list = list(diag.get("injected_eps_coord_topq", []) or [])
                tq = int(len(theta_list))
                if tq <= 0:
                    raise RuntimeError("theta_coord_topq is empty; cannot build eigenvalue table")

                key = (str(dataset_name), str(kernel_cfg.get("name", "")), float(eps), int(mtot_meta), int(tq))
                if key in _LAMBDA1_A_EIGSH_CACHE:
                    lam_list = list(_LAMBDA1_A_EIGSH_CACHE[key])
                else:
                    lam_list = _eigvals_A_via_cupy_eigsh(out.backend, out.data_ctx, float(REG_LAMBDA), top_q=tq)
                    _LAMBDA1_A_EIGSH_CACHE[key] = list(lam_list)
                if key in _LAMBDA_DEFAULT_V3_CACHE:
                    lam_default_list = list(_LAMBDA_DEFAULT_V3_CACHE[key])
                else:
                    lam_default_list = _eigvals_A_via_default_v3(out.backend, out.data_ctx, float(REG_LAMBDA), top_q=tq)
                    _LAMBDA_DEFAULT_V3_CACHE[key] = list(lam_default_list)

                rows_ev = []
                for i in range(tq):
                    lam = float(lam_list[i]) if i < len(lam_list) else float("nan")
                    lam_default = float(lam_default_list[i]) if i < len(lam_default_list) else float("nan")
                    th = float(theta_list[i])
                    ratio = float(lam / th) if np.isfinite(lam) and np.isfinite(th) and th != 0.0 else np.nan
                    inj = float(eps_list[i]) if i < len(eps_list) else np.nan
                    rows_ev.append(
                        {
                            "i": i + 1,
                            "lambda_i(A)": _fmt_sci_sig3(lam),
                            "lambda_i(default_v3)": _fmt_sci_sig3(lam_default),
                            "theta_i(W)": _fmt_sci_sig3(th),
                            "ratio": ratio,
                            "injected_residual": inj,
                        }
                    )

                ev_df = pd.DataFrame(rows_ev)
                print(
                    f"[EIGVAL_TABLE] dataset={dataset_name} kernel={kernel_cfg['name']} eps={eps:g} "
                    f"q={tq} precond_kind=coordinate_nystrom"
                )
                display(ev_df)
            except Exception as _e:
                print(f"[EIGVAL_TABLE] skipped/failed: {type(_e).__name__}: {_e}")

        t_exact_v1 = float(row.get("t_original_exact_gpu_precompute_v1_s", np.nan))
        if not np.isfinite(t_exact_v1):
            t_exact_v1 = float(row.get("time_precompute_NUFFT", np.nan))
        t_binned = float(row.get("time_precompute_binned", np.nan))
        if pcm_lc in ("c0", "c1", "c2"):
            row["time_precompute"] = float(t_binned)
            row["time_precompute_NUFFT"] = float(np.nan)
        else:
            row["time_precompute"] = float(t_exact_v1)
            row["time_precompute_NUFFT"] = float(t_exact_v1)

        tp_p = row.get("time_precompute")
        tp_s = row.get("time_solve")
        tp_r = row.get("time_predict")
        if tp_p is None or tp_s is None or (isinstance(tp_p, float) and np.isnan(tp_p)) or (isinstance(tp_s, float) and np.isnan(tp_s)):
            row["time_train"] = np.nan
        else:
            row["time_train"] = float(
                float(tp_p)
                + float(row.get("time_eigenspace") or 0.0)
                + float(row.get("time_precond_build") or 0.0)
                + float(tp_s)
            )
        if not np.isfinite(float(row.get("time_train", np.nan))) or tp_r is None or (isinstance(tp_r, float) and np.isnan(tp_r)):
            row["wall_s_total"] = np.nan
        else:
            row["wall_s_total"] = float(row["time_train"]) + float(tp_r)
        row["case_label"] = _label_case(mode, top_q, pcm_lc, eps, kernel_cfg["name"])
        return row
    finally:
        _BENCHMARK_PC_METHOD_ACTIVE = None
        _LAST_PC_PATCH_EXTRA.clear()
        _clear_state(clear_pool=bool(BENCHMARK_AFTER_CASE_GPU_POOL_FLUSH))


def run_warmup(dataset_payload: dict) -> None:
    if not RUN_WARMUP:
        return
    x_train = np.asarray(dataset_payload["x_train"], dtype=np.float64)
    y_train = np.asarray(dataset_payload["y_train"], dtype=np.float64)
    x_test = np.asarray(dataset_payload["x_test"], dtype=np.float64)
    y_test = np.asarray(dataset_payload["y_test"], dtype=np.float64)
    n_warm = min(int(WARMUP_TRAIN_SAMPLES), int(x_train.shape[0]))
    if n_warm < 8:
        return
    warm_payload = dict(dataset_payload)
    warm_payload["x_train"] = np.asarray(x_train[:n_warm], dtype=np.float64)
    warm_payload["y_train"] = np.asarray(y_train[:n_warm], dtype=np.float64)
    warm_payload["x_test"] = np.asarray(x_test[: min(len(x_test), max(8, min(256, len(x_test))))], dtype=np.float64)
    warm_payload["y_test"] = np.asarray(y_test[: min(len(y_test), max(8, min(256, len(y_test))))], dtype=np.float64)
    active_precompute_methods = _resolve_precompute_methods(dataset_payload)

    print(
        f"warmup dataset={dataset_payload['name']} n_train={n_warm} "
        f"active_precompute_methods={active_precompute_methods}"
    )
    for kernel_cfg in KERNEL_SPECS:
        for eps, mode_spec, pcm in itertools.product(EPS_LIST, MODE_SPECS, active_precompute_methods):
            print(f"  warmup: kernel={kernel_cfg['name']} eps={eps:g} mode={mode_spec['mode']} q={int(mode_spec['top_q'])} pcm={pcm}")
            _run_case_once(warm_payload, kernel_cfg, eps, mode_spec, pcm, repeat_idx=-1, warmup_only=True)


def run_benchmark(dataset_payloads: list[dict]) -> pd.DataFrame:
    rows = []
    for dataset_payload in dataset_payloads:
        dataset_paths = _dataset_output_paths(str(dataset_payload["name"]))
        dataset_rows = []
        active_precompute_methods = _resolve_precompute_methods(dataset_payload)
        print("=" * 100)
        print(
            f"dataset={dataset_payload['name']} | dim={dataset_payload['dim']} | "
            f"n_train={dataset_payload['n_train']} | n_test={dataset_payload['n_test']} | "
            f"active_precompute_methods={active_precompute_methods}"
        )
        for kernel_cfg in KERNEL_SPECS:
            for eps in EPS_LIST:
                for mode_spec in MODE_SPECS:
                    for pcm in active_precompute_methods:
                        print("-" * 100)
                        print(
                            f"kernel={kernel_cfg['name']} eps={eps:g} mode={mode_spec['mode']} "
                            f"q={int(mode_spec['top_q'])} pcm={pcm} repeats={REPEATS}"
                        )
                        for rep in range(int(REPEATS)):
                            try:
                                row = _run_case_once(dataset_payload, kernel_cfg, eps, mode_spec, pcm, repeat_idx=rep)
                                rows.append(row)
                                dataset_rows.append(row)
                                print(
                                    f"ok rep={rep:02d} dataset={dataset_payload['name']} mode={row['mode']} pcm={row['precompute_method']} "
                                    f"q={row['top_q']} train={row.get('time_train', np.nan):.4f}s wall={row.get('wall_s_total', np.nan):.4f}s "
                                    f"iters={row.get('cg_iters', -1)} rmse={row.get('rmse_test', np.nan):.6e}"
                                )
                            except Exception as e:
                                err_row = {
                                    "run_id": f"{RUN_TAG}_{dataset_payload['name']}_{kernel_cfg['name']}_{mode_spec['mode']}_pcm{str(pcm).lower()}_q{int(mode_spec['top_q'])}_eps{eps:g}_rep{rep}",
                                    "timestamp": datetime.now().isoformat(),
                                    "dataset": dataset_payload['name'],
                                    "dataset_tag": str(dataset_paths['dataset_tag']),
                                    "dataset_path": str(dataset_payload['path']),
                                    "dataset_output_dir": str(dataset_paths['dataset_dir']),
                                    "dim": int(dataset_payload['dim']),
                                    "n_total": int(dataset_payload['n_total']),
                                    "n_train": int(dataset_payload['n_train']),
                                    "n_test": int(dataset_payload['n_test']),
                                    "kernel_name": str(kernel_cfg['name']),
                                    "kernel_family": str(kernel_cfg.get('family', 'matern')),
                                    "kernel_lengthscale": float(kernel_cfg['lengthscale']),
                                    "kernel_nu": float(kernel_cfg.get('nu', np.nan)) if kernel_cfg.get('nu', None) is not None else np.nan,
                                    "kernel_variance": float(kernel_cfg.get('variance', 1.0)),
                                    "eps": float(eps),
                                    "mode": str(mode_spec['mode']),
                                    "method_variant": str(mode_spec.get('method_variant', mode_spec.get('nystrom_variant', mode_spec.get('mode', '')))),
                                    "eig_method_requested": str(mode_spec.get('eig_method', 'subspace_iter' if str(mode_spec.get('mode', '')) in ('gpu_v3_topq', 'gpu_v3_custom_topq') else ('eigenpro_nystrom' if str(mode_spec.get('mode', '')) == 'gpu_v3_topq_eigenpro_nystrom' else ''))),
                                    "eig_method_effective": str(mode_spec.get('eig_method', 'subspace_iter' if str(mode_spec.get('mode', '')) in ('gpu_v3_topq', 'gpu_v3_custom_topq') else ('eigenpro_nystrom' if str(mode_spec.get('mode', '')) == 'gpu_v3_topq_eigenpro_nystrom' else ''))),
                                    "top_q": int(mode_spec.get('top_q', 0)),
                                    "precompute_method": str(pcm).strip().lower(),
                                    "reg_lambda": float(REG_LAMBDA),
                                    "cg_tol": float(SOLVE_TOL),
                                    "status": "error",
                                    "error": f"{type(e).__name__}: {e}",
                                    "repeat_idx": int(rep),
                                    "repeat_count": int(REPEATS),
                                    "case_label": _label_case(str(mode_spec['mode']), int(mode_spec.get('top_q', 0)), str(pcm).lower(), float(eps), kernel_cfg['name']),
                                }
                                rows.append(err_row)
                                dataset_rows.append(err_row)
                                traceback.print_exc()
        dataset_raw_df = pd.DataFrame(dataset_rows)
        dataset_raw_df.to_csv(dataset_paths['raw_csv'], index=False)
        pd.DataFrame(rows).to_csv(GLOBAL_RAW_CSV, index=False)
        print(f"dataset raw csv saved: {dataset_paths['raw_csv']} rows={len(dataset_rows)}")
        print(f"global raw csv saved: {GLOBAL_RAW_CSV} rows={len(rows)}")
    raw_df = pd.DataFrame(rows)
    raw_df.to_csv(GLOBAL_RAW_CSV, index=False)
    return raw_df


def summarize_benchmark(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    if raw_df.empty:
        return pd.DataFrame(), {}
    ok_df = raw_df[raw_df["status"] == "ok"].copy()
    if ok_df.empty:
        return pd.DataFrame(), {}

    group_cols = [
        "dataset",
        "dataset_tag",
        "dim",
        "n_total",
        "n_train",
        "n_test",
        "kernel_name",
        "kernel_family",
        "kernel_lengthscale",
        "kernel_nu",
        "kernel_variance",
        "eps",
        "mode",
        "method_variant",
        "eig_method_requested",
        "eig_method_effective",
        "top_q",
        "precompute_method",
        # Keep Nyström variants separated in summary (no merging across refine/precond branches).
        "nystrom_variant",
        "eigenpro_nystrom_precond_kind_requested",
        "eigenpro_nystrom_precond_kind_effective",
        "eigenpro_nystrom_refine_mode_requested",
        "eigenpro_nystrom_refine_mode_effective",
        "eigenpro_nystrom_refine_iters_requested",
        "case_label",
    ]
    metrics = [
        "rmse_train",
        "rmse_test",
        "mae_test",
        "r2_test",
        "time_precompute",
        "time_precompute_NUFFT",
        "time_precompute_binned",
        "time_eigenspace",
        "time_precond_build",
        "time_solve",
        "time_predict",
        "time_train",
        "wall_s_total",
        "cg_iters",
        "cg_relres",
        "t_matvec_total",
        "t_precond_total",
        "peak_mem_gb",
        "eig_nystrom_kernel_s",
        "effective_work_ratio",
        "t_original_exact_gpu_precompute_v1_s",
        "t_h2d_xy_s",
        "t_gpu_fused_bin_moments_s",
        "t_compact_occupied_s",
        "t_binned_cufinufft_on_centers_s",
        "t_rhs_D_multiply_s",
    ]

    for c in group_cols:
        if c not in raw_df.columns:
            raw_df[c] = np.nan
        if c not in ok_df.columns:
            ok_df[c] = np.nan
    for m in metrics:
        if m not in raw_df.columns:
            raw_df[m] = np.nan
        if m not in ok_df.columns:
            ok_df[m] = np.nan

    def _quantile(series: pd.Series, q: float) -> float:
        s = pd.to_numeric(series, errors="coerce")
        if s.notna().sum() == 0:
            return float("nan")
        return float(s.quantile(q))

    gb = ok_df.groupby(group_cols, dropna=False)
    parts = []
    for m in metrics:
        part = gb[m].agg(["median", "mean", "std"]).reset_index()
        part = part.rename(columns={"median": f"{m}_median", "mean": f"{m}_mean", "std": f"{m}_std"})
        q10 = gb[m].apply(lambda s: _quantile(s, 0.10)).reset_index(name=f"{m}_p10")
        q90 = gb[m].apply(lambda s: _quantile(s, 0.90)).reset_index(name=f"{m}_p90")
        part = part.merge(q10, on=group_cols, how="left").merge(q90, on=group_cols, how="left")
        parts.append(part)

    summary_df = parts[0]
    for part in parts[1:]:
        keep_cols = [c for c in part.columns if c not in group_cols]
        summary_df = summary_df.merge(part[group_cols + keep_cols], on=group_cols, how="left")

    count_df = raw_df.groupby(group_cols, as_index=False).agg(
        repeat_count=("run_id", "count"),
        fail_count=("status", lambda x: int((x != "ok").sum())),
    )
    summary_df = summary_df.merge(count_df, on=group_cols, how="left")
    summary_df = summary_df.sort_values([
        "dataset",
        "kernel_name",
        "eps",
        "mode",
        "method_variant",
        "eig_method_effective",
        "top_q",
        "precompute_method",
        "nystrom_variant",
        "eigenpro_nystrom_precond_kind_effective",
        "eigenpro_nystrom_refine_mode_effective",
    ]).reset_index(drop=True)
    summary_df.to_csv(GLOBAL_SUMMARY_CSV, index=False)

    per_dataset_summary = {}
    for dataset_name, dataset_df in summary_df.groupby("dataset", dropna=False):
        dataset_paths = _dataset_output_paths(str(dataset_name))
        dataset_df = dataset_df.reset_index(drop=True)
        dataset_df.to_csv(dataset_paths["summary_csv"], index=False)
        per_dataset_summary[str(dataset_name)] = dataset_df

    return summary_df, per_dataset_summary

## SLQ

In [ ]:
# ---- Prepare benchmark outputs before paper figures ----
if "dataset_payloads" not in globals() or len(globals().get("dataset_payloads", [])) == 0:
    dataset_specs = build_dataset_specs()
    if len(dataset_specs) == 0:
        raise ValueError(f"No processed dataset files found under {PROCESSED_DATA_DIR}")

    env_info = _collect_env_info(dataset_specs)
    GLOBAL_ENV_JSON.write_text(json.dumps(env_info, indent=2), encoding="utf-8")
    print("global env info saved:", GLOBAL_ENV_JSON)

    dataset_payloads = [load_dataset(spec) for spec in dataset_specs]
    for payload in dataset_payloads:
        dataset_paths = _dataset_output_paths(str(payload["name"]))
        dataset_env = {
            "run_tag": RUN_TAG,
            "dataset": payload["name"],
            "dataset_tag": dataset_paths["dataset_tag"],
            "dataset_path": payload["path"],
            "metadata_path": payload.get("metadata_path", ""),
            "dim": int(payload["dim"]),
            "n_total": int(payload["n_total"]),
            "n_train": int(payload["n_train"]),
            "n_test": int(payload["n_test"]),
            "available_arrays": payload.get("available_arrays", []),
            "source_metadata": payload.get("metadata", {}),
            "kernel_specs": KERNEL_SPECS,
            "eps_list": list(EPS_LIST),
            "mode_specs": MODE_SPECS,
            "precompute_methods": PRECOMPUTE_METHODS,
            "reg_lambda": REG_LAMBDA,
            "solve_tol": SOLVE_TOL,
            "gpu_maxiter": GPU_MAXITER,
            "gpu_nufft": GPU_NUFFT,
            "l2_scaled": L2_SCALED,
            "enable_slq": bool(ENABLE_SLQ),
        }
        dataset_paths["env_json"].write_text(json.dumps(dataset_env, indent=2), encoding="utf-8")
        print(
            f"loaded dataset={payload['name']} | dim={payload['dim']} | "
            f"n_total={payload['n_total']} | n_train={payload['n_train']} | n_test={payload['n_test']} | "
            f"dataset_dir={dataset_paths['dataset_dir']}"
        )
else:
    print("dataset_payloads already prepared.")

if "raw_df" not in globals() or "summary_df" not in globals():
    if RUN_WARMUP and len(dataset_payloads) > 0:
        run_warmup(dataset_payloads[0])
    raw_df = run_benchmark(dataset_payloads)
    summary_df, per_dataset_summary = summarize_benchmark(raw_df)
else:
    print("raw_df / summary_df already prepared.")

print("benchmark outputs ready for paper figures.")

# Paper Figures and Tables

本节直接基于本 notebook 已生成的 `raw_df`、`summary_df` 和 `dataset_payloads`，整理主文图表。

主文保留：

- `Table 1`: experiment setup summary
- `Fig. 1`: synthetic scaling
- `Fig. 2`: synthetic preconditioning effect
- `Fig. 3`: synthetic accuracy stability
- `Fig. 4`: USGS task visualization
- `Fig. 5`: USGS scaling and speedup
- `Table 2`: Greengard-style timing / accuracy table

选择规则：

- `EFGP-CG (q=0, original)` 固定使用 `gpu_v1_topq0` 且 `precompute_method=original`
- `V3-topq>0 best (original only)` 只在默认 `gpu_v3_topq` eigenspace 方法内比较，并固定 `precompute_method=original`
- `V3-topq>0 best (C1 only)` 只在默认 `gpu_v3_topq` eigenspace 方法内比较，并固定 `precompute_method=C1`
- `Eigenmethod best (all enabled)` 在所有启用的 eigenspace / Nystr"om / custom 方法中比较，允许使用全部已开启的 `precompute_method`
- 每个 `N_train` 都按 `time_train` 最小选出对应 series 的 best case
- 所有图结束后，统一列出每张图实际选中的方法、`q` 和 `precompute_method`

附录建议（本节先不展开）：q sweep、memory、kernel/lengthscale sweep、spatial block split。

In [ ]:
from IPython.display import Markdown, display

PAPER_MAIN_N_VALUES = [100_000, 300_000, 1_000_000, 3_000_000, 10_000_000, 30_000_000, 100_000_000]
PAPER_TABLE2_N_VALUES = [100_000, 1_000_000, 10_000_000, 100_000_000]
FIXED_PREDICT_TEST_SIZE = 100_000
VIS_SAMPLE_TRAIN = 200_000
VIS_SAMPLE_TEST = 100_000
PAPER_SELECTIONS: dict[str, pd.DataFrame] = {}
PAPER_ARTIFACTS_DIR = OUT_DIR / "paper_artifacts"
PAPER_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def _fmt_n_short(n: int) -> str:
    n = int(n)
    if n >= 1_000_000:
        val = n / 1_000_000
        return f"{val:g}M"
    if n >= 1_000:
        val = n / 1_000
        return f"{val:g}k"
    return str(n)



def _dataset_family(name: str) -> str:
    s = str(name)
    if s.startswith("synthetic_true_func_2d_"):
        return "synthetic"
    if s.startswith("USGS_LPC_IL_Winnebago_2018_ground_elevation_regression"):
        return "usgs"
    return "other"



PAPER_SERIES_Q0 = "EFGP-CG (q=0, original)"
PAPER_SERIES_V3_ORIGINAL = "V3-topq>0 best (original only)"
PAPER_SERIES_V3_C1 = "V3-topq>0 best (C1 only)"
PAPER_SERIES_EIG = "Eigenmethod best (all enabled)"



def _series_label(row: pd.Series) -> str:
    mode = str(row.get("mode", ""))
    top_q = int(row.get("top_q", 0)) if pd.notna(row.get("top_q", np.nan)) else 0
    method_variant = str(row.get("method_variant", ""))
    pcm = str(row.get("precompute_method", ""))
    if mode == "gpu_v1_topq0":
        return f"gpu_v1_topq0 | q=0 | pcm={pcm}"
    return f"{mode} | {method_variant} | q={top_q} | pcm={pcm}"



def _prep_paper_results(raw_df: pd.DataFrame) -> pd.DataFrame:
    if raw_df is None or raw_df.empty:
        raise ValueError("raw_df is empty. Please run the benchmark cells before generating paper figures.")

    ok_df = raw_df[raw_df["status"] == "ok"].copy()
    if ok_df.empty:
        raise ValueError("raw_df has no successful runs. Cannot build paper figures.")

    ok_df["dataset_family"] = ok_df["dataset"].map(_dataset_family)
    ok_df = ok_df[ok_df["dataset_family"].isin(["synthetic", "usgs"])].copy()
    if ok_df.empty:
        raise ValueError("No synthetic / usgs successful rows found in raw_df.")

    numeric_cols = [
        "n_train", "n_test", "dim", "eps", "top_q", "grid_mtot", "grid_M",
        "rmse_test", "mae_test", "r2_test", "time_precompute", "time_eigenspace",
        "time_precond_build", "time_solve", "time_predict", "time_train", "wall_s_total",
        "cg_iters", "cg_relres", "peak_mem_gb", "kernel_lengthscale", "kernel_nu",
        "kernel_variance", "reg_lambda", "time_precompute_NUFFT", "time_precompute_binned",
    ]
    for col in numeric_cols:
        if col in ok_df.columns:
            ok_df[col] = pd.to_numeric(ok_df[col], errors="coerce")

    group_cols = [
        "dataset_family", "dataset", "dim", "n_total", "n_train", "n_test",
        "kernel_name", "kernel_family", "kernel_lengthscale", "kernel_nu", "kernel_variance",
        "eps", "mode", "mode_requested", "mode_effective", "method_variant",
        "eig_method_requested", "eig_method_effective", "top_q", "precompute_method",
        "nystrom_variant", "eigenpro_nystrom_precond_kind_effective",
        "eigenpro_nystrom_refine_mode_effective", "eigenpro_nystrom_refine_iters_requested",
        "grid_mtot", "grid_M", "case_label",
    ]
    group_cols = [c for c in group_cols if c in ok_df.columns]

    metric_cols = [
        "rmse_test", "mae_test", "r2_test", "time_precompute", "time_eigenspace",
        "time_precond_build", "time_solve", "time_predict", "time_train", "wall_s_total",
        "cg_iters", "cg_relres", "peak_mem_gb", "time_precompute_NUFFT", "time_precompute_binned",
    ]
    metric_cols = [c for c in metric_cols if c in ok_df.columns]

    agg_df = ok_df.groupby(group_cols, dropna=False)[metric_cols].median().reset_index()
    agg_df["time_eig_total"] = (
        pd.to_numeric(agg_df.get("time_eigenspace", np.nan), errors="coerce").fillna(0.0)
        + pd.to_numeric(agg_df.get("time_precond_build", np.nan), errors="coerce").fillna(0.0)
    )
    agg_df["predict_fixed_100k_est"] = np.where(
        pd.to_numeric(agg_df["n_test"], errors="coerce") > 0,
        pd.to_numeric(agg_df.get("time_predict", np.nan), errors="coerce") * FIXED_PREDICT_TEST_SIZE / pd.to_numeric(agg_df["n_test"], errors="coerce"),
        np.nan,
    )
    agg_df["method_desc"] = agg_df.apply(_series_label, axis=1)
    return agg_df.sort_values(["dataset_family", "n_train", "time_train", "top_q"]).reset_index(drop=True)



def _pick_fastest_per_n(df: pd.DataFrame, mask: pd.Series) -> pd.DataFrame:
    sub = df[mask].copy()
    if sub.empty:
        return sub
    sub = sub[np.isfinite(pd.to_numeric(sub["time_train"], errors="coerce"))].copy()
    if sub.empty:
        return sub
    idx = sub.groupby(["dataset_family", "n_train"], dropna=False)["time_train"].idxmin()
    return sub.loc[idx].sort_values(["dataset_family", "n_train"]).reset_index(drop=True)



def _pcm_mask(df: pd.DataFrame, value: str) -> pd.Series:
    return df["precompute_method"].astype(str).str.strip().str.lower() == str(value).strip().lower()



def _collect_series(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    q0 = _pick_fastest_per_n(
        df,
        (df["mode"] == "gpu_v1_topq0")
        & (df["mode_requested"] == "gpu_v1_topq0")
        & (df["mode_effective"] == "gpu_v1_topq0")
        & _pcm_mask(df, "original"),
    )
    v3_original = _pick_fastest_per_n(
        df,
        (df["mode"] == "gpu_v3_topq")
        & (df["mode_requested"] == "gpu_v3_topq")
        & (df["mode_effective"] == "gpu_v3_topq")
        & (pd.to_numeric(df["top_q"], errors="coerce") > 0)
        & _pcm_mask(df, "original"),
    )
    v3_c1 = _pick_fastest_per_n(
        df,
        (df["mode"] == "gpu_v3_topq")
        & (df["mode_requested"] == "gpu_v3_topq")
        & (df["mode_effective"] == "gpu_v3_topq")
        & (pd.to_numeric(df["top_q"], errors="coerce") > 0)
        & _pcm_mask(df, "c1"),
    )
    eig_best = _pick_fastest_per_n(
        df,
        (df["mode_requested"].isin(["gpu_v3_topq_eigenpro_nystrom", "gpu_v3_custom_topq"]))
        & (df["mode_effective"] == df["mode_requested"]),
    )
    return {
        PAPER_SERIES_Q0: q0,
        PAPER_SERIES_V3_ORIGINAL: v3_original,
        PAPER_SERIES_V3_C1: v3_c1,
        PAPER_SERIES_EIG: eig_best,
    }



def _subset_series_map(selection_map: dict[str, pd.DataFrame], series_names: list[str]) -> dict[str, pd.DataFrame]:
    return {name: selection_map.get(name, pd.DataFrame()).copy() for name in series_names}



def _record_selection(fig_name: str, selection_map: dict[str, pd.DataFrame], family: str | None = None) -> None:
    rows = []
    for series_name, df in selection_map.items():
        sub = df.copy()
        if family is not None:
            sub = sub[sub["dataset_family"] == family].copy()
        if sub.empty:
            continue
        sub["figure"] = fig_name
        sub["series"] = series_name
        rows.append(
            sub[
                [
                    "figure", "series", "dataset_family", "dataset", "n_train", "n_test",
                    "mode", "mode_requested", "mode_effective", "method_variant",
                    "eig_method_effective", "top_q", "precompute_method", "time_train",
                    "time_solve", "cg_iters", "rmse_test", "mae_test", "r2_test", "method_desc",
                ]
            ].copy()
        )
    PAPER_SELECTIONS[fig_name] = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()



def _plot_series(ax, selection_map: dict[str, pd.DataFrame], family: str, y_col: str, *, logx: bool = True, logy: bool = False, ylabel: str = "", title: str = "") -> None:
    style_map = {
        PAPER_SERIES_Q0: dict(marker="o", linewidth=2.0),
        PAPER_SERIES_V3_ORIGINAL: dict(marker="s", linewidth=2.0),
        PAPER_SERIES_V3_C1: dict(marker="D", linewidth=2.0),
        PAPER_SERIES_EIG: dict(marker="^", linewidth=2.2),
    }
    plotted = False
    for series_name, df in selection_map.items():
        sub = df[df["dataset_family"] == family].copy()
        if sub.empty or y_col not in sub.columns:
            continue
        sub = sub[np.isfinite(pd.to_numeric(sub[y_col], errors="coerce"))].sort_values("n_train")
        if sub.empty:
            continue
        ax.plot(sub["n_train"], sub[y_col], label=series_name, **style_map.get(series_name, {}))
        plotted = True
    if logx:
        ax.set_xscale("log")
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("N_train")
    ax.set_ylabel(ylabel or y_col)
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.25)
    if plotted:
        ax.legend(fontsize=9)



def _sample_points(x: np.ndarray, y: np.ndarray, n_keep: int, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    if x.shape[0] <= n_keep:
        return np.asarray(x), np.asarray(y)
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(x.shape[0], size=int(n_keep), replace=False)
    return np.asarray(x[idx]), np.asarray(y[idx])



def _binned_mean_image(x: np.ndarray, y: np.ndarray, bins: int = 256) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    x0 = x[:, 0]
    x1 = x[:, 1]
    w_sum, xe, ye = np.histogram2d(x0, x1, bins=bins, weights=y)
    count, _, _ = np.histogram2d(x0, x1, bins=[xe, ye])
    img = np.divide(w_sum, count, out=np.full_like(w_sum, np.nan, dtype=np.float64), where=count > 0)
    return img.T, xe, ye



def _binned_density_image(x: np.ndarray, bins: int = 256) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = np.asarray(x, dtype=np.float64)
    count, xe, ye = np.histogram2d(x[:, 0], x[:, 1], bins=bins)
    return np.log10(count.T + 1.0), xe, ye



def _family_payload_rows(dataset_payloads: list[dict]) -> pd.DataFrame:
    rows = []
    for payload in dataset_payloads:
        meta = payload.get("metadata", {}) or {}
        rows.append(
            {
                "dataset": payload["name"],
                "dataset_family": _dataset_family(payload["name"]),
                "n_train": int(payload["n_train"]),
                "n_test": int(payload["n_test"]),
                "dim": int(payload["dim"]),
                "task": str(meta.get("paper_task_statement", meta.get("task_type", ""))),
                "split_method": str((meta.get("split", {}) or {}).get("method", "")),
                "test_size": (meta.get("split", {}) or {}).get("test_size", np.nan),
            }
        )
    return pd.DataFrame(rows)



def _n_list_string(vals: list[int]) -> str:
    return ", ".join(_fmt_n_short(v) for v in sorted(int(v) for v in vals))



def _range_or_single(vals: list[int | float], formatter=str) -> str:
    clean = [v for v in vals if pd.notna(v)]
    if len(clean) == 0:
        return "n/a"
    uniq = sorted({formatter(v) for v in clean})
    if len(uniq) == 1:
        return uniq[0]
    return f"{uniq[0]} -- {uniq[-1]}"



def _latex_table(df: pd.DataFrame, caption: str, label: str) -> str:
    return df.to_latex(index=False, escape=True, caption=caption, label=label)



def _save_table_bundle(base_name: str, df: pd.DataFrame, *, caption: str = "", label: str = "") -> dict[str, Path]:
    csv_path = PAPER_ARTIFACTS_DIR / f"{base_name}.csv"
    md_path = PAPER_ARTIFACTS_DIR / f"{base_name}.md"
    tex_path = PAPER_ARTIFACTS_DIR / f"{base_name}.tex"
    df.to_csv(csv_path, index=False)
    try:
        md_text = df.to_markdown(index=False)
    except Exception:
        md_text = df.to_string(index=False)
    md_path.write_text(md_text, encoding="utf-8")
    tex_text = _latex_table(df, caption or base_name, label or base_name)
    tex_path.write_text(tex_text, encoding="utf-8")
    return {"csv": csv_path, "md": md_path, "tex": tex_path}



def _save_figure_bundle(fig, base_name: str, *, dpi: int = 220) -> dict[str, Path]:
    png_path = PAPER_ARTIFACTS_DIR / f"{base_name}.png"
    pdf_path = PAPER_ARTIFACTS_DIR / f"{base_name}.pdf"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    return {"png": png_path, "pdf": pdf_path}



def _save_selection_bundle(base_name: str, df: pd.DataFrame) -> dict[str, Path]:
    csv_path = PAPER_ARTIFACTS_DIR / f"{base_name}.csv"
    md_path = PAPER_ARTIFACTS_DIR / f"{base_name}.md"
    df.to_csv(csv_path, index=False)
    try:
        md_text = df.to_markdown(index=False)
    except Exception:
        md_text = df.to_string(index=False)
    md_path.write_text(md_text, encoding="utf-8")
    return {"csv": csv_path, "md": md_path}



def _resolve_paper_runtime_inputs() -> tuple[pd.DataFrame, list[dict]]:
    raw_df_eff = globals().get("raw_df", None)
    dataset_payloads_eff = globals().get("dataset_payloads", None)

    if raw_df_eff is None:
        raise NameError(
            "raw_df is not defined. Please run the cell 'Prepare benchmark outputs before paper figures' first."
        )
    if dataset_payloads_eff is None:
        raise NameError(
            "dataset_payloads is not defined. Please run the cell 'Prepare benchmark outputs before paper figures' first."
        )

    if not isinstance(raw_df_eff, pd.DataFrame):
        raise TypeError(f"raw_df must be a pandas DataFrame, got {type(raw_df_eff)}")
    if not isinstance(dataset_payloads_eff, list):
        raise TypeError(f"dataset_payloads must be a list, got {type(dataset_payloads_eff)}")

    return raw_df_eff, dataset_payloads_eff


In [ ]:
raw_df_eff, dataset_payloads_eff = _resolve_paper_runtime_inputs()
paper_df = _prep_paper_results(raw_df_eff)
payload_df = _family_payload_rows(dataset_payloads_eff)
series_map = _collect_series(paper_df)
PAPER_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print("paper artifacts dir:", PAPER_ARTIFACTS_DIR)

# ----------------------------------------------------------------------------
# Table 1: experiment setup summary
# ----------------------------------------------------------------------------
exp_rows = []
active_q_values = sorted({int(s.get("top_q", 0)) for s in MODE_SPECS if int(s.get("top_q", 0)) >= 0})
active_q_string = ", ".join(str(q) for q in active_q_values)
kernel_string = ", ".join(
    f"{str(k.get('family', 'matern')).capitalize()}(ls={k['lengthscale']}, nu={k.get('nu', 'n/a')})"
    for k in KERNEL_SPECS
)

for family in ["synthetic", "usgs"]:
    fam_payload = payload_df[payload_df["dataset_family"] == family].copy()
    fam_results = paper_df[paper_df["dataset_family"] == family].copy()
    if fam_payload.empty:
        continue

    n_train_vals = sorted(fam_payload["n_train"].astype(int).unique().tolist())
    n_test_vals = sorted(fam_payload["n_test"].astype(int).unique().tolist())
    grid_mtot_vals = sorted(pd.to_numeric(fam_results.get("grid_mtot", np.nan), errors="coerce").dropna().astype(int).unique().tolist())
    grid_halfm_vals = sorted({int((mtot - 1) // 2) for mtot in grid_mtot_vals if int(mtot) > 0})
    grid_M_vals = sorted(pd.to_numeric(fam_results.get("grid_M", np.nan), errors="coerce").dropna().astype(int).unique().tolist())

    if family == "synthetic":
        dataset_name = "Synthetic"
        n_test_protocol = f"fixed test set: {_range_or_single(n_test_vals, _fmt_n_short)}"
        task = fam_payload["task"].iloc[0] or "(x1, x2) -> f(x)"
    else:
        dataset_name = "USGS 3DEP LiDAR"
        n_test_protocol = f"random 80/20 split; N_test = {_n_list_string(n_test_vals)}"
        task = fam_payload["task"].iloc[0] or "(x, y) -> elevation"

    exp_rows.append(
        {
            "Dataset": dataset_name,
            "Task": task,
            "d": int(fam_payload["dim"].iloc[0]),
            "N_train values": _n_list_string(n_train_vals),
            "N_test protocol": n_test_protocol,
            "Kernel": kernel_string,
            "lengthscale": ", ".join(str(k["lengthscale"]) for k in KERNEL_SPECS),
            "lambda": REG_LAMBDA,
            "epsilon": EPS_LIST[0] if len(EPS_LIST) == 1 else str(EPS_LIST),
            "m": _range_or_single(grid_halfm_vals, lambda v: str(int(v))),
            "M=(2m+1)^d": _range_or_single(grid_M_vals, lambda v: _fmt_n_short(int(v))),
            "q values": active_q_string,
            "preconditioner type": "q=0 CG (original only) / positive-q V3 best on original / positive-q V3 best on C1 / eigenspace-custom best with all enabled",
        }
    )

paper_table1_df = pd.DataFrame(exp_rows)
paper_table1_caption = "Summary of experimental settings."
paper_table1_label = "tab:exp-setup"
paper_table1_paths = _save_table_bundle("table1_experiment_setup", paper_table1_df, caption=paper_table1_caption, label=paper_table1_label)
display(Markdown("## Table 1. Experiment Setup Summary"))
display(paper_table1_df)
print(_latex_table(paper_table1_df, paper_table1_caption, paper_table1_label))
print("saved Table 1:", paper_table1_paths)

# ----------------------------------------------------------------------------
# Fig. 1: synthetic scaling
# ----------------------------------------------------------------------------
fig1, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
fig1_metrics = [
    ("time_precompute", "precompute time (s)"),
    ("time_eig_total", "eigenspace/setup time (s)"),
    ("time_solve", "solve time (s)"),
    ("time_train", "total train time (s)"),
]
for ax, (metric, ylabel) in zip(axes.ravel(), fig1_metrics):
    _plot_series(ax, series_map, "synthetic", metric, logx=True, logy=True, ylabel=ylabel, title=ylabel)
fig1.suptitle("Fig. 1 Synthetic scaling: timing vs N_train", fontsize=15)
fig1_paths = _save_figure_bundle(fig1, "fig1_synthetic_scaling")
plt.show()
print("saved Fig. 1:", fig1_paths)
_record_selection("Fig. 1", series_map, family="synthetic")
plt.close(fig1)

# ----------------------------------------------------------------------------
# Fig. 2: synthetic preconditioning effect
# ----------------------------------------------------------------------------
fig2, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
_plot_series(axes[0], series_map, "synthetic", "cg_iters", logx=True, logy=False, ylabel="CG iterations", title="(a) CG iterations vs N_train")
_plot_series(axes[1], series_map, "synthetic", "time_solve", logx=True, logy=True, ylabel="solve time (s)", title="(b) Solve time vs N_train")
fig2.suptitle("Fig. 2 Synthetic preconditioning effect", fontsize=15)
fig2_paths = _save_figure_bundle(fig2, "fig2_synthetic_preconditioning")
plt.show()
print("saved Fig. 2:", fig2_paths)
_record_selection("Fig. 2", series_map, family="synthetic")
plt.close(fig2)

# ----------------------------------------------------------------------------
# Fig. 3: synthetic accuracy stability
# ----------------------------------------------------------------------------
fig3, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
_plot_series(axes[0], series_map, "synthetic", "rmse_test", logx=True, logy=True, ylabel="RMSE on noiseless y_test", title="(a) RMSE vs N_train")
_plot_series(axes[1], series_map, "synthetic", "mae_test", logx=True, logy=True, ylabel="MAE on noiseless y_test", title="(b) MAE vs N_train")
fig3.suptitle("Fig. 3 Synthetic accuracy stability", fontsize=15)
fig3_paths = _save_figure_bundle(fig3, "fig3_synthetic_accuracy_stability")
plt.show()
print("saved Fig. 3:", fig3_paths)
_record_selection("Fig. 3", series_map, family="synthetic")
plt.close(fig3)

# ----------------------------------------------------------------------------
# Fig. 4: USGS task visualization
# ----------------------------------------------------------------------------
usgs_payloads = [p for p in dataset_payloads_eff if _dataset_family(p["name"]) == "usgs"]
if len(usgs_payloads) == 0:
    raise ValueError("No USGS payload available for Fig. 4.")

vis_payload = max(
    usgs_payloads,
    key=lambda p: (int(p.get("n_train", 0)) + int(p.get("n_test", 0)), int(p.get("n_train", 0))),
)
x_train_vis, y_train_vis = _sample_points(np.asarray(vis_payload["x_train"]), np.asarray(vis_payload["y_train"]), VIS_SAMPLE_TRAIN, seed=0)
x_test_vis, y_test_vis = _sample_points(np.asarray(vis_payload["x_test"]), np.asarray(vis_payload["y_test"]), VIS_SAMPLE_TEST, seed=1)
all_xy_vis = np.vstack([x_train_vis, x_test_vis])
train_mean_img, xe_t, ye_t = _binned_mean_image(x_train_vis, y_train_vis, bins=256)
test_mean_img, xe_te, ye_te = _binned_mean_image(x_test_vis, y_test_vis, bins=256)
density_img, xe_d, ye_d = _binned_density_image(all_xy_vis, bins=256)

fig4, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
im0 = axes[0].imshow(train_mean_img, origin="lower", cmap="terrain", aspect="auto")
axes[0].set_title("(a) Train elevation map")
axes[0].set_xlabel("x bin")
axes[0].set_ylabel("y bin")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].scatter(x_train_vis[:, 0], x_train_vis[:, 1], s=1, alpha=0.25, label="train")
axes[1].scatter(x_test_vis[:, 0], x_test_vis[:, 1], s=1, alpha=0.25, label="test")
axes[1].set_title("(b) Train / test locations")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend(markerscale=4)

im2 = axes[2].imshow(test_mean_img, origin="lower", cmap="viridis", aspect="auto")
axes[2].set_title("(c) Held-out target elevation map")
axes[2].set_xlabel("x bin")
axes[2].set_ylabel("y bin")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

fig4.suptitle(
    f"Fig. 4 USGS LiDAR task visualization (dataset={vis_payload['name']}, sampled train={len(x_train_vis):,}, sampled test={len(x_test_vis):,})",
    fontsize=15,
)
fig4_paths = _save_figure_bundle(fig4, "fig4_usgs_task_visualization")
plt.show()
print("saved Fig. 4:", fig4_paths)
PAPER_SELECTIONS["Fig. 4"] = pd.DataFrame(
    [{
        "figure": "Fig. 4",
        "series": "visualization dataset",
        "dataset_family": "usgs",
        "dataset": vis_payload["name"],
        "n_train": int(vis_payload["n_train"]),
        "n_test": int(vis_payload["n_test"]),
        "mode": "visualization_only",
        "mode_requested": "visualization_only",
        "mode_effective": "visualization_only",
        "method_variant": "none",
        "eig_method_effective": "none",
        "top_q": np.nan,
        "precompute_method": "none",
        "time_train": np.nan,
        "time_solve": np.nan,
        "cg_iters": np.nan,
        "rmse_test": np.nan,
        "mae_test": np.nan,
        "r2_test": np.nan,
        "method_desc": f"USGS visualization payload: {vis_payload['name']}",
    }]
)
plt.close(fig4)

# ----------------------------------------------------------------------------
# Fig. 5: USGS scaling and speedup
# ----------------------------------------------------------------------------
_record_selection("Fig. 5", series_map, family="usgs")
fig5, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

usgs_eig_best = series_map[PAPER_SERIES_EIG]
usgs_eig_best = usgs_eig_best[usgs_eig_best["dataset_family"] == "usgs"].sort_values("n_train")
if usgs_eig_best.empty:
    raise ValueError("No USGS eigenspace-best rows found for Fig. 5.")

x_labels = [_fmt_n_short(v) for v in usgs_eig_best["n_train"].astype(int).tolist()]
x_pos = np.arange(len(x_labels))
axes[0, 0].bar(x_pos, usgs_eig_best["time_precompute"], label="precompute")
axes[0, 0].bar(x_pos, usgs_eig_best["time_eig_total"], bottom=usgs_eig_best["time_precompute"], label="eig/setup")
axes[0, 0].bar(x_pos, usgs_eig_best["time_solve"], bottom=usgs_eig_best["time_precompute"] + usgs_eig_best["time_eig_total"], label="solve")
axes[0, 0].bar(
    x_pos,
    usgs_eig_best["predict_fixed_100k_est"],
    bottom=usgs_eig_best["time_precompute"] + usgs_eig_best["time_eig_total"] + usgs_eig_best["time_solve"],
    label=f"predict@{_fmt_n_short(FIXED_PREDICT_TEST_SIZE)} (est.)",
)
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(x_labels, rotation=30)
axes[0, 0].set_ylabel("seconds")
axes[0, 0].set_title("(a) Runtime breakdown of best eigenspace method")
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, axis="y", alpha=0.25)

_plot_series(axes[0, 1], series_map, "usgs", "cg_iters", logx=True, logy=False, ylabel="CG iterations", title="(b) CG iterations vs N_train")

q0_usgs = series_map[PAPER_SERIES_Q0]
q0_usgs = q0_usgs[q0_usgs["dataset_family"] == "usgs"][["n_train", "time_solve", "time_train"]].rename(columns={"time_solve": "solve_q0", "time_train": "train_q0"})
speed_series_specs = [
    (PAPER_SERIES_V3_ORIGINAL, "solve_v3_orig", "train_v3_orig", "s", "V3 original-best / q=0"),
    (PAPER_SERIES_V3_C1, "solve_v3_c1", "train_v3_c1", "D", "V3 C1-best / q=0"),
    (PAPER_SERIES_EIG, "solve_eig", "train_eig", "^", "eig-best / q=0"),
]
for series_name, solve_col, train_col, marker, label in speed_series_specs:
    sub = series_map[series_name]
    sub = sub[sub["dataset_family"] == "usgs"][["n_train", "time_solve", "time_train"]].rename(columns={"time_solve": solve_col, "time_train": train_col})
    speed_df = q0_usgs.merge(sub, on="n_train", how="inner").sort_values("n_train")
    if speed_df.empty:
        continue
    axes[1, 0].plot(speed_df["n_train"], speed_df["solve_q0"] / speed_df[solve_col], marker=marker, linewidth=2.0, label=f"solve speedup: {label}")
    axes[1, 0].plot(speed_df["n_train"], speed_df["train_q0"] / speed_df[train_col], marker=marker, linestyle="--", linewidth=1.8, label=f"train speedup: {label}")
axes[1, 0].set_xscale("log")
axes[1, 0].set_xlabel("N_train")
axes[1, 0].set_ylabel("speedup")
axes[1, 0].set_title("(c) Solve / train speedup vs q=0")
axes[1, 0].grid(True, which="both", alpha=0.25)
axes[1, 0].legend(fontsize=8)

ax_rmse = axes[1, 1]
ax_r2 = ax_rmse.twinx()
for series_name, df in series_map.items():
    sub = df[df["dataset_family"] == "usgs"].sort_values("n_train")
    if sub.empty:
        continue
    ax_rmse.plot(sub["n_train"], sub["rmse_test"], marker="o", linewidth=2.0, label=f"{series_name} RMSE")
    ax_r2.plot(sub["n_train"], sub["r2_test"], marker="x", linestyle="--", linewidth=1.6, label=f"{series_name} R2")
ax_rmse.set_xscale("log")
ax_rmse.set_yscale("log")
ax_rmse.set_xlabel("N_train")
ax_rmse.set_ylabel("RMSE_test")
ax_r2.set_ylabel("R2_test")
ax_rmse.set_title("(d) Accuracy stability on USGS")
ax_rmse.grid(True, which="both", alpha=0.25)
lines1, labels1 = ax_rmse.get_legend_handles_labels()
lines2, labels2 = ax_r2.get_legend_handles_labels()
ax_rmse.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="best")

fig5.suptitle("Fig. 5 USGS scaling and speedup", fontsize=15)
fig5_paths = _save_figure_bundle(fig5, "fig5_usgs_scaling_speedup")
plt.show()
print("saved Fig. 5:", fig5_paths)
plt.close(fig5)

# ----------------------------------------------------------------------------
# Table 2: Greengard-style timing / accuracy table
# ----------------------------------------------------------------------------
table2_parts = []
for series_name in [PAPER_SERIES_Q0, PAPER_SERIES_V3_ORIGINAL, PAPER_SERIES_V3_C1, PAPER_SERIES_EIG]:
    sub = series_map[series_name]
    sub = sub[(sub["dataset_family"] == "usgs") & (sub["n_train"].isin(PAPER_TABLE2_N_VALUES))].copy()
    if sub.empty:
        continue
    sub["Series"] = series_name
    table2_parts.append(sub)
paper_table2_df = pd.concat(table2_parts, ignore_index=True) if table2_parts else pd.DataFrame()
paper_table2_df = paper_table2_df.sort_values(["n_train", "Series"]).reset_index(drop=True)
paper_table2_df["N_train"] = paper_table2_df["n_train"].astype(int).map(_fmt_n_short)
paper_table2_df["N_test"] = paper_table2_df["n_test"].astype(int).map(_fmt_n_short)
paper_table2_df["M"] = paper_table2_df["grid_M"].astype(float).round().astype("Int64")
paper_table2_df["q"] = paper_table2_df["top_q"].astype(float).round().astype("Int64")
paper_table2_df["precompute"] = paper_table2_df["time_precompute"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df["eig/setup"] = paper_table2_df["time_eig_total"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df["solve"] = paper_table2_df["time_solve"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df[f"predict@{_fmt_n_short(FIXED_PREDICT_TEST_SIZE)}"] = paper_table2_df["predict_fixed_100k_est"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df["total_train"] = paper_table2_df["time_train"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df["CG iters"] = paper_table2_df["cg_iters"].round().astype("Int64")
paper_table2_df["RMSE"] = paper_table2_df["rmse_test"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df["MAE"] = paper_table2_df["mae_test"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df["R2"] = paper_table2_df["r2_test"].map(lambda v: float(v) if pd.notna(v) else np.nan)
paper_table2_df["peak_mem_gb"] = paper_table2_df["peak_mem_gb"].map(lambda v: float(v) if pd.notna(v) else np.nan)

table2_display_cols = [
    "Series", "N_train", "N_test", "M", "q", "precompute", "eig/setup", "solve",
    f"predict@{_fmt_n_short(FIXED_PREDICT_TEST_SIZE)}", "total_train", "CG iters", "RMSE", "MAE", "R2", "peak_mem_gb",
]
paper_table2_caption = f"Greengard-style timing / accuracy table on USGS. Prediction time is rescaled to a fixed {_fmt_n_short(FIXED_PREDICT_TEST_SIZE)}-point test subset from the recorded full-test timing."
paper_table2_label = "tab:usgs-greengard"
paper_table2_paths = _save_table_bundle(
    "table2_usgs_greengard",
    paper_table2_df[table2_display_cols],
    caption=paper_table2_caption,
    label=paper_table2_label,
)
display(Markdown("## Table 2. Greengard-style Timing / Accuracy Table"))
display(paper_table2_df[table2_display_cols])
print(
    _latex_table(
        paper_table2_df[table2_display_cols],
        paper_table2_caption,
        paper_table2_label,
    )
)
print("saved Table 2:", paper_table2_paths)

# ----------------------------------------------------------------------------
# Method selection listing after all figures
# ----------------------------------------------------------------------------
display(Markdown("## Method Choices Used in Each Figure"))
method_choice_rows = []
for fig_name in ["Fig. 1", "Fig. 2", "Fig. 3", "Fig. 4", "Fig. 5"]:
    sel_df = PAPER_SELECTIONS.get(fig_name, pd.DataFrame())
    if sel_df is None or sel_df.empty:
        continue
    display(Markdown(f"### {fig_name}"))
    show_cols = [
        "series", "dataset", "n_train", "n_test", "mode_requested", "mode_effective",
        "method_variant", "top_q", "precompute_method", "time_train", "time_solve", "cg_iters", "method_desc",
    ]
    show_cols = [c for c in show_cols if c in sel_df.columns]
    shown_df = sel_df.sort_values([c for c in ["series", "n_train"] if c in sel_df.columns])[show_cols].reset_index(drop=True)
    display(shown_df)
    shown_df = shown_df.copy()
    shown_df.insert(0, "figure", fig_name)
    method_choice_rows.append(shown_df)

if len(method_choice_rows) > 0:
    method_choices_df = pd.concat(method_choice_rows, ignore_index=True)
    method_choice_paths = _save_selection_bundle("method_choices_by_figure", method_choices_df)
    print("saved method choices:", method_choice_paths)

# ----------------------------------------------------------------------------
# Reduced comparison set: q=0 original + eigenspace best only
# ----------------------------------------------------------------------------
display(Markdown("## Reduced Comparison Set: q=0 Original vs Eigenmethod Best"))
reduced_series_map = _subset_series_map(series_map, [PAPER_SERIES_Q0, PAPER_SERIES_EIG])
reduced_artifacts = {}
reduced_method_choice_rows = []

reduced_table1_df = paper_table1_df.copy()
reduced_table1_df["preconditioner type"] = "q=0 CG (original only) / eigenspace-custom best with all enabled"
reduced_table1_caption = "Summary of experimental settings for the reduced two-series comparison."
reduced_table1_label = "tab:exp-setup-reduced"
reduced_table1_paths = _save_table_bundle(
    "table1_experiment_setup_reduced_q0_eig",
    reduced_table1_df,
    caption=reduced_table1_caption,
    label=reduced_table1_label,
)
reduced_artifacts["table1"] = reduced_table1_paths
display(Markdown("## Table 1 (Reduced). Experiment Setup Summary"))
display(reduced_table1_df)
print(_latex_table(reduced_table1_df, reduced_table1_caption, reduced_table1_label))
print("saved reduced Table 1:", reduced_table1_paths)

reduced_fig1, reduced_axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for ax, (metric, ylabel) in zip(reduced_axes.ravel(), fig1_metrics):
    _plot_series(ax, reduced_series_map, "synthetic", metric, logx=True, logy=True, ylabel=ylabel, title=ylabel)
reduced_fig1.suptitle("Fig. 1 (Reduced) Synthetic scaling: timing vs N_train", fontsize=15)
reduced_fig1_paths = _save_figure_bundle(reduced_fig1, "fig1_synthetic_scaling_reduced_q0_eig")
reduced_artifacts["fig1"] = reduced_fig1_paths
plt.show()
print("saved reduced Fig. 1:", reduced_fig1_paths)
_record_selection("Reduced Fig. 1", reduced_series_map, family="synthetic")
plt.close(reduced_fig1)

reduced_fig2, reduced_axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
_plot_series(reduced_axes[0], reduced_series_map, "synthetic", "cg_iters", logx=True, logy=False, ylabel="CG iterations", title="(a) CG iterations vs N_train")
_plot_series(reduced_axes[1], reduced_series_map, "synthetic", "time_solve", logx=True, logy=True, ylabel="solve time (s)", title="(b) Solve time vs N_train")
reduced_fig2.suptitle("Fig. 2 (Reduced) Synthetic preconditioning effect", fontsize=15)
reduced_fig2_paths = _save_figure_bundle(reduced_fig2, "fig2_synthetic_preconditioning_reduced_q0_eig")
reduced_artifacts["fig2"] = reduced_fig2_paths
plt.show()
print("saved reduced Fig. 2:", reduced_fig2_paths)
_record_selection("Reduced Fig. 2", reduced_series_map, family="synthetic")
plt.close(reduced_fig2)

reduced_fig3, reduced_axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
_plot_series(reduced_axes[0], reduced_series_map, "synthetic", "rmse_test", logx=True, logy=True, ylabel="RMSE on noiseless y_test", title="(a) RMSE vs N_train")
_plot_series(reduced_axes[1], reduced_series_map, "synthetic", "mae_test", logx=True, logy=True, ylabel="MAE on noiseless y_test", title="(b) MAE vs N_train")
reduced_fig3.suptitle("Fig. 3 (Reduced) Synthetic accuracy stability", fontsize=15)
reduced_fig3_paths = _save_figure_bundle(reduced_fig3, "fig3_synthetic_accuracy_stability_reduced_q0_eig")
reduced_artifacts["fig3"] = reduced_fig3_paths
plt.show()
print("saved reduced Fig. 3:", reduced_fig3_paths)
_record_selection("Reduced Fig. 3", reduced_series_map, family="synthetic")
plt.close(reduced_fig3)

display(Markdown("## Fig. 4 (Reduced)"))
print("Fig. 4 is method-independent USGS task visualization, so the reduced two-series version reuses the main Fig. 4 artifact.")
reduced_artifacts["fig4"] = {k: v for k, v in fig4_paths.items()}
PAPER_SELECTIONS["Reduced Fig. 4"] = PAPER_SELECTIONS["Fig. 4"].copy()

_record_selection("Reduced Fig. 5", reduced_series_map, family="usgs")
reduced_fig5, reduced_axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
reduced_usgs_eig = reduced_series_map[PAPER_SERIES_EIG]
reduced_usgs_eig = reduced_usgs_eig[reduced_usgs_eig["dataset_family"] == "usgs"].sort_values("n_train")
if reduced_usgs_eig.empty:
    raise ValueError("No USGS eigenspace-best rows found for reduced Fig. 5.")
reduced_x_labels = [_fmt_n_short(v) for v in reduced_usgs_eig["n_train"].astype(int).tolist()]
reduced_x_pos = np.arange(len(reduced_x_labels))
reduced_axes[0, 0].bar(reduced_x_pos, reduced_usgs_eig["time_precompute"], label="precompute")
reduced_axes[0, 0].bar(reduced_x_pos, reduced_usgs_eig["time_eig_total"], bottom=reduced_usgs_eig["time_precompute"], label="eig/setup")
reduced_axes[0, 0].bar(reduced_x_pos, reduced_usgs_eig["time_solve"], bottom=reduced_usgs_eig["time_precompute"] + reduced_usgs_eig["time_eig_total"], label="solve")
reduced_axes[0, 0].bar(
    reduced_x_pos,
    reduced_usgs_eig["predict_fixed_100k_est"],
    bottom=reduced_usgs_eig["time_precompute"] + reduced_usgs_eig["time_eig_total"] + reduced_usgs_eig["time_solve"],
    label=f"predict@{_fmt_n_short(FIXED_PREDICT_TEST_SIZE)} (est.)",
)
reduced_axes[0, 0].set_xticks(reduced_x_pos)
reduced_axes[0, 0].set_xticklabels(reduced_x_labels, rotation=30)
reduced_axes[0, 0].set_ylabel("seconds")
reduced_axes[0, 0].set_title("(a) Runtime breakdown of best eigenspace method")
reduced_axes[0, 0].legend(fontsize=9)
reduced_axes[0, 0].grid(True, axis="y", alpha=0.25)
_plot_series(reduced_axes[0, 1], reduced_series_map, "usgs", "cg_iters", logx=True, logy=False, ylabel="CG iterations", title="(b) CG iterations vs N_train")
reduced_q0_usgs = reduced_series_map[PAPER_SERIES_Q0]
reduced_q0_usgs = reduced_q0_usgs[reduced_q0_usgs["dataset_family"] == "usgs"][["n_train", "time_solve", "time_train"]].rename(columns={"time_solve": "solve_q0", "time_train": "train_q0"})
reduced_speed_sub = reduced_series_map[PAPER_SERIES_EIG]
reduced_speed_sub = reduced_speed_sub[reduced_speed_sub["dataset_family"] == "usgs"][["n_train", "time_solve", "time_train"]].rename(columns={"time_solve": "solve_eig", "time_train": "train_eig"})
reduced_speed_df = reduced_q0_usgs.merge(reduced_speed_sub, on="n_train", how="inner").sort_values("n_train")
if not reduced_speed_df.empty:
    reduced_axes[1, 0].plot(reduced_speed_df["n_train"], reduced_speed_df["solve_q0"] / reduced_speed_df["solve_eig"], marker="^", linewidth=2.2, label="solve speedup: eig-best / q=0")
    reduced_axes[1, 0].plot(reduced_speed_df["n_train"], reduced_speed_df["train_q0"] / reduced_speed_df["train_eig"], marker="^", linestyle="--", linewidth=1.8, label="train speedup: eig-best / q=0")
reduced_axes[1, 0].set_xscale("log")
reduced_axes[1, 0].set_xlabel("N_train")
reduced_axes[1, 0].set_ylabel("speedup")
reduced_axes[1, 0].set_title("(c) Solve / train speedup vs q=0")
reduced_axes[1, 0].grid(True, which="both", alpha=0.25)
reduced_axes[1, 0].legend(fontsize=8)
reduced_ax_rmse = reduced_axes[1, 1]
reduced_ax_r2 = reduced_ax_rmse.twinx()
for series_name, df in reduced_series_map.items():
    sub = df[df["dataset_family"] == "usgs"].sort_values("n_train")
    if sub.empty:
        continue
    reduced_ax_rmse.plot(sub["n_train"], sub["rmse_test"], marker="o", linewidth=2.0, label=f"{series_name} RMSE")
    reduced_ax_r2.plot(sub["n_train"], sub["r2_test"], marker="x", linestyle="--", linewidth=1.6, label=f"{series_name} R2")
reduced_ax_rmse.set_xscale("log")
reduced_ax_rmse.set_yscale("log")
reduced_ax_rmse.set_xlabel("N_train")
reduced_ax_rmse.set_ylabel("RMSE_test")
reduced_ax_r2.set_ylabel("R2_test")
reduced_ax_rmse.set_title("(d) Accuracy stability on USGS")
reduced_ax_rmse.grid(True, which="both", alpha=0.25)
reduced_lines1, reduced_labels1 = reduced_ax_rmse.get_legend_handles_labels()
reduced_lines2, reduced_labels2 = reduced_ax_r2.get_legend_handles_labels()
reduced_ax_rmse.legend(reduced_lines1 + reduced_lines2, reduced_labels1 + reduced_labels2, fontsize=7, loc="best")
reduced_fig5.suptitle("Fig. 5 (Reduced) USGS scaling and speedup", fontsize=15)
reduced_fig5_paths = _save_figure_bundle(reduced_fig5, "fig5_usgs_scaling_speedup_reduced_q0_eig")
reduced_artifacts["fig5"] = reduced_fig5_paths
plt.show()
print("saved reduced Fig. 5:", reduced_fig5_paths)
plt.close(reduced_fig5)

reduced_table2_parts = []
for series_name in [PAPER_SERIES_Q0, PAPER_SERIES_EIG]:
    sub = reduced_series_map[series_name]
    sub = sub[(sub["dataset_family"] == "usgs") & (sub["n_train"].isin(PAPER_TABLE2_N_VALUES))].copy()
    if sub.empty:
        continue
    sub["Series"] = series_name
    reduced_table2_parts.append(sub)
reduced_paper_table2_df = pd.concat(reduced_table2_parts, ignore_index=True) if reduced_table2_parts else pd.DataFrame()
reduced_paper_table2_df = reduced_paper_table2_df.sort_values(["n_train", "Series"]).reset_index(drop=True)
reduced_paper_table2_df["N_train"] = reduced_paper_table2_df["n_train"].astype(int).map(_fmt_n_short)
reduced_paper_table2_df["N_test"] = reduced_paper_table2_df["n_test"].astype(int).map(_fmt_n_short)
reduced_paper_table2_df["M"] = reduced_paper_table2_df["grid_M"].astype(float).round().astype("Int64")
reduced_paper_table2_df["q"] = reduced_paper_table2_df["top_q"].astype(float).round().astype("Int64")
reduced_paper_table2_df["precompute"] = reduced_paper_table2_df["time_precompute"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df["eig/setup"] = reduced_paper_table2_df["time_eig_total"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df["solve"] = reduced_paper_table2_df["time_solve"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df[f"predict@{_fmt_n_short(FIXED_PREDICT_TEST_SIZE)}"] = reduced_paper_table2_df["predict_fixed_100k_est"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df["total_train"] = reduced_paper_table2_df["time_train"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df["CG iters"] = reduced_paper_table2_df["cg_iters"].round().astype("Int64")
reduced_paper_table2_df["RMSE"] = reduced_paper_table2_df["rmse_test"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df["MAE"] = reduced_paper_table2_df["mae_test"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df["R2"] = reduced_paper_table2_df["r2_test"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_paper_table2_df["peak_mem_gb"] = reduced_paper_table2_df["peak_mem_gb"].map(lambda v: float(v) if pd.notna(v) else np.nan)
reduced_table2_paths = _save_table_bundle(
    "table2_usgs_greengard_reduced_q0_eig",
    reduced_paper_table2_df[table2_display_cols],
    caption="Greengard-style timing / accuracy table on USGS for the reduced two-series comparison.",
    label="tab:usgs-greengard-reduced",
)
reduced_artifacts["table2"] = reduced_table2_paths
display(Markdown("## Table 2 (Reduced). Greengard-style Timing / Accuracy Table"))
display(reduced_paper_table2_df[table2_display_cols])
print(_latex_table(reduced_paper_table2_df[table2_display_cols], "Greengard-style timing / accuracy table on USGS for the reduced two-series comparison.", "tab:usgs-greengard-reduced"))
print("saved reduced Table 2:", reduced_table2_paths)

display(Markdown("## Reduced Method Choices Used in Each Figure"))
for fig_name in ["Reduced Fig. 1", "Reduced Fig. 2", "Reduced Fig. 3", "Reduced Fig. 4", "Reduced Fig. 5"]:
    sel_df = PAPER_SELECTIONS.get(fig_name, pd.DataFrame())
    if sel_df is None or sel_df.empty:
        continue
    display(Markdown(f"### {fig_name}"))
    show_cols = [
        "series", "dataset", "n_train", "n_test", "mode_requested", "mode_effective",
        "method_variant", "top_q", "precompute_method", "time_train", "time_solve", "cg_iters", "method_desc",
    ]
    show_cols = [c for c in show_cols if c in sel_df.columns]
    shown_df = sel_df.sort_values([c for c in ["series", "n_train"] if c in sel_df.columns])[show_cols].reset_index(drop=True)
    display(shown_df)
    shown_df = shown_df.copy()
    shown_df.insert(0, "figure", fig_name)
    reduced_method_choice_rows.append(shown_df)

if len(reduced_method_choice_rows) > 0:
    reduced_method_choices_df = pd.concat(reduced_method_choice_rows, ignore_index=True)
    reduced_method_choice_paths = _save_selection_bundle("method_choices_by_figure_reduced_q0_eig", reduced_method_choices_df)
    reduced_artifacts["method_choices"] = reduced_method_choice_paths
    print("saved reduced method choices:", reduced_method_choice_paths)

paper_artifact_manifest = {
    "paper_artifacts_dir": str(PAPER_ARTIFACTS_DIR),
    "table1": {k: str(v) for k, v in paper_table1_paths.items()},
    "fig1": {k: str(v) for k, v in fig1_paths.items()},
    "fig2": {k: str(v) for k, v in fig2_paths.items()},
    "fig3": {k: str(v) for k, v in fig3_paths.items()},
    "fig4": {k: str(v) for k, v in fig4_paths.items()},
    "fig5": {k: str(v) for k, v in fig5_paths.items()},
    "table2": {k: str(v) for k, v in paper_table2_paths.items()},
}
if len(method_choice_rows) > 0:
    paper_artifact_manifest["method_choices"] = {k: str(v) for k, v in method_choice_paths.items()}
paper_artifact_manifest["reduced_q0_eig"] = {
    key: ({k: str(v) for k, v in val.items()} if isinstance(val, dict) else str(val))
    for key, val in reduced_artifacts.items()
}

paper_artifact_manifest_path = PAPER_ARTIFACTS_DIR / "paper_artifact_manifest.json"
paper_artifact_manifest_path.write_text(json.dumps(paper_artifact_manifest, indent=2), encoding="utf-8")
print("saved paper artifact manifest:", paper_artifact_manifest_path)


In [ ]:
# ---- Optional SLQ + execution ----
slq_diag = None
slq_pcg_spectrum = None
SLQLanczosConfig = None
run_slq_lanczos_diagnostic = None
summarize_slq_diagnostics = None
save_slq_plots = None
build_slq_matvec_for_benchmark_mode = None


def _sanitize_tag(s: str) -> str:
    return re.sub(r"[^0-9a-zA-Z_\-]+", "_", str(s)).strip("_")


def _spec_key(spec: dict) -> tuple[str, int, str, str]:
    return (
        str(spec.get("mode", "")),
        int(spec.get("top_q", -1)),
        str(spec.get("method_variant", spec.get("nystrom_variant", ""))),
        str(spec.get("eig_method", "")),
    )


def _pick_mode_specs(all_specs: list[dict], selected_specs: list[dict]) -> list[dict]:
    if len(selected_specs) == 0:
        return list(all_specs)
    out = []
    seen = set()
    all_by_key = {_spec_key(s): s for s in all_specs}
    for spec in selected_specs:
        key = _spec_key(spec)
        use_spec = dict(all_by_key[key]) if key in all_by_key else dict(spec)
        if key not in seen:
            out.append(use_spec)
            seen.add(key)
    return out


def _pick_dataset_payloads(dataset_payloads: list[dict], selected_names: list[str]) -> list[dict]:
    if len(selected_names) == 0:
        return list(dataset_payloads)
    selected = set(selected_names)
    return [p for p in dataset_payloads if p["name"] in selected]


def _to_jsonable(obj):
    if is_dataclass(obj):
        return _to_jsonable(asdict(obj))
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    return obj


def _q_lookup(q_map: dict, q: float) -> float:
    if not isinstance(q_map, dict):
        return float("nan")
    if q in q_map:
        return float(q_map[q])
    key = str(q)
    if key in q_map:
        return float(q_map[key])
    for k, v in q_map.items():
        try:
            if abs(float(k) - float(q)) < 1e-15:
                return float(v)
        except Exception:
            pass
    return float("nan")


def _normalize_summary_schema(summary: dict) -> tuple[dict, dict, dict]:
    if isinstance(summary, dict) and all(k in summary for k in ("raw", "derived", "views")):
        return summary.get("raw", {}), summary.get("derived", {}), summary.get("views", {})
    return {}, {}, {}


def _default_slq_mode_specs(all_specs: list[dict]) -> list[dict]:
    # 只保留两组：
    # 1) q=0 baseline -> gpu_v1_topq0
    # 2) 默认 V3 的最大 q -> gpu_v3_topq, q=max(V3_TOPQ_LIST)
    out = []
    q0_candidates = [s for s in all_specs if str(s.get("mode", "")) == "gpu_v1_topq0"]
    if len(q0_candidates) > 0:
        out.append(dict(q0_candidates[0]))

    v3_positive = [
        s for s in all_specs
        if str(s.get("mode", "")) == "gpu_v3_topq" and int(s.get("top_q", 0)) > 0
    ]
    if len(v3_positive) > 0:
        max_q = max(int(s.get("top_q", 0)) for s in v3_positive)
        best_v3_max = [s for s in v3_positive if int(s.get("top_q", 0)) == int(max_q)]
        if len(best_v3_max) > 0:
            out.append(dict(best_v3_max[0]))
    return out


def _default_slq_dataset_names(dataset_payloads: list[dict]) -> list[str]:
    # synthetic / usgs 各自只取最小 N_train 的一个数据集。
    payloads = list(dataset_payloads)
    if len(payloads) == 0:
        return []

    family_groups: dict[str, list[dict]] = {}
    for payload in payloads:
        name = str(payload.get("name", ""))
        if name.startswith("synthetic_true_func_2d_"):
            family = "synthetic"
        elif name.startswith("USGS_LPC_IL_Winnebago_2018_ground_elevation_regression"):
            family = "usgs"
        else:
            family = "other"
        family_groups.setdefault(family, []).append(payload)

    selected = []
    for family in ("synthetic", "usgs"):
        group = family_groups.get(family, [])
        if len(group) == 0:
            continue
        group_sorted = sorted(group, key=lambda p: (int(p.get("n_train", 10**30)), str(p.get("name", ""))))
        selected.append(str(group_sorted[0]["name"]))
    return selected


def run_selected_slq(dataset_payloads: list[dict]) -> pd.DataFrame:
    global slq_diag, slq_pcg_spectrum, SLQLanczosConfig, run_slq_lanczos_diagnostic
    global summarize_slq_diagnostics, save_slq_plots, build_slq_matvec_for_benchmark_mode

    if not ENABLE_SLQ:
        return pd.DataFrame()

    import efgp_eigenpro_py.gpu.slq_diagnostics as slq_diag_mod
    import efgp_eigenpro_py.gpu.slq_pcg_spectrum as slq_pcg_mod

    slq_diag = importlib.reload(slq_diag_mod)
    slq_pcg_spectrum = importlib.reload(slq_pcg_mod)
    SLQLanczosConfig = slq_diag.SLQLanczosConfig
    run_slq_lanczos_diagnostic = slq_diag.run_slq_lanczos_diagnostic
    summarize_slq_diagnostics = slq_diag.summarize_slq_diagnostics
    save_slq_plots = slq_diag.save_slq_plots
    build_slq_matvec_for_benchmark_mode = slq_pcg_spectrum.build_slq_matvec_for_benchmark_mode

    selected_datasets_eff = list(SLQ_SELECTED_DATASETS)
    if len(selected_datasets_eff) == 0 and bool(SLQ_USE_MIN_N_PER_FAMILY):
        selected_datasets_eff = _default_slq_dataset_names(dataset_payloads)

    selected_mode_specs_eff = list(SLQ_SELECTED_MODE_SPECS)
    if len(selected_mode_specs_eff) == 0 and bool(SLQ_USE_DEFAULT_V3_ONLY):
        selected_mode_specs_eff = _default_slq_mode_specs(MODE_SPECS)

    picked_payloads = _pick_dataset_payloads(dataset_payloads, selected_datasets_eff)
    picked_specs = _pick_mode_specs(MODE_SPECS, selected_mode_specs_eff)
    picked_specs = [s for s in picked_specs if str(s.get("mode", "")) != "gpu_v3_topq_eigenpro_nystrom"]
    if len(picked_payloads) == 0 or len(picked_specs) == 0:
        print("SKIP SLQ: no eligible dataset or mode selected.")
        return pd.DataFrame()

    print("SLQ selected datasets:", [str(p["name"]) for p in picked_payloads])
    print(
        "SLQ selected modes:",
        [f"{str(s.get('mode', ''))}:q={int(s.get('top_q', 0))}" for s in picked_specs],
    )

    slq_cfg = SLQLanczosConfig(**SLQ_CFG_KWARGS)
    prefix_steps = list(range(SLQ_PREFIX_STEP, int(slq_cfg.k_max) + 1, SLQ_PREFIX_STEP))
    rows = []

    for case_idx, dataset_payload in enumerate(picked_payloads, start=1):
        dataset_paths = _dataset_output_paths(str(dataset_payload["name"]))
        dataset_slq_dir = dataset_paths["slq_dir"]
        dataset_slq_dir.mkdir(parents=True, exist_ok=True)
        x_train = np.asarray(dataset_payload["x_train"], dtype=np.float64)
        y_train = np.asarray(dataset_payload["y_train"], dtype=np.float64)
        for spec in picked_specs:
            mode = str(spec.get("mode", ""))
            top_q = int(spec.get("top_q", 0))
            dim = int(dataset_payload["dim"])
            kernel_cfg = KERNEL_SPECS[0]
            kernel = _make_kernel(kernel_cfg, dim)
            solver = EFGPSolver(kernel=kernel, reg_lambda=REG_LAMBDA, eps=float(EPS_LIST[0]), nufft_tol=1e-10, l2scaled=L2_SCALED)
            cfg = GPURunConfig(reg_lambda=REG_LAMBDA, tol=SOLVE_TOL, maxiter=GPU_MAXITER, chunk_size=None, debug_finite_checks=False, backend=BackendConfig(nufft=GPU_NUFFT))
            case_seed = int(SLQ_SEED_BASE + case_idx)
            backend, matvec, size, slq_op_meta = build_slq_matvec_for_benchmark_mode(
                mode,
                solver,
                x_train,
                y_train,
                cfg,
                top_q=top_q,
                combo_cfg=None,
                v3_oversample=V3_OVERSAMPLE,
                v3_n_iter=V3_N_ITER,
                dim=dim,
            )
            spec_mode = str(slq_op_meta.get("slq_spectrum", "")) if isinstance(slq_op_meta, dict) else ""
            spectrum_mode = "hermitian" if spec_mode == "M_inv_A" else SLQ_SUMMARY_MODE

            print(f"[SLQ] dataset={dataset_payload['name']} mode={mode} q={top_q} size={size}")
            t0 = time.perf_counter()
            slq_res = run_slq_lanczos_diagnostic(backend=backend, matvec=matvec, size=size, cfg=slq_cfg)
            summary = summarize_slq_diagnostics(slq_res, prefix_steps=prefix_steps, spectrum_mode=spectrum_mode)
            t1 = time.perf_counter()
            if isinstance(summary, dict) and isinstance(summary.get("raw"), dict) and isinstance(slq_op_meta, dict):
                summary["raw"]["slq_operator_meta"] = _to_jsonable(slq_op_meta)

            raw_part, derived_part, views_part = _normalize_summary_schema(summary)
            case_tag = f"{_sanitize_tag(dataset_payload['name'])}_{_sanitize_tag(mode)}_q{top_q}_seed{case_seed}"
            case_dir = dataset_slq_dir / case_tag
            case_dir.mkdir(parents=True, exist_ok=True)
            (case_dir / "slq_summary.json").write_text(json.dumps(_to_jsonable(summary), indent=2), encoding="utf-8")
            np.savez_compressed(case_dir / "lanczos_coeffs.npz", alpha=slq_res.alpha, beta=slq_res.beta, active_steps=slq_res.active_steps)
            save_slq_plots(summary, str(case_dir / "plots"), dpi=160)

            q_map = derived_part.get("final_quantiles", {}) if isinstance(derived_part, dict) else {}
            rows.append(
                {
                    "dataset": dataset_payload["name"],
                    "mode": mode,
                    "top_q": top_q,
                    "precompute_method": str(SLQ_SELECTED_PRECOMPUTE_METHOD).lower(),
                    "size": int(size),
                    "wall_s": float(t1 - t0),
                    "lambda_hat_min": float(derived_part.get("lambda_hat_min", np.nan)),
                    "lambda_hat_max": float(derived_part.get("lambda_hat_max", np.nan)),
                    "q01": _q_lookup(q_map, 0.01),
                    "q99": _q_lookup(q_map, 0.99),
                    "health": str(views_part.get("headline", {}).get("health", "")) if isinstance(views_part, dict) else "",
                    "dominant_issue": str(views_part.get("headline", {}).get("dominant_issue", "")) if isinstance(views_part, dict) else "",
                    "out_dir": str(case_dir),
                }
            )
    slq_df = pd.DataFrame(rows)
    if not slq_df.empty:
        for dataset_name, dataset_slq_df in slq_df.groupby("dataset", dropna=False):
            dataset_paths = _dataset_output_paths(str(dataset_name))
            dataset_paths["slq_dir"].mkdir(parents=True, exist_ok=True)
            dataset_slq_df.reset_index(drop=True).to_csv(dataset_paths["slq_dir"] / "slq_cases_summary.csv", index=False)
    return slq_df


if "dataset_payloads" not in globals() or len(globals().get("dataset_payloads", [])) == 0:
    dataset_specs = build_dataset_specs()
    if len(dataset_specs) == 0:
        raise ValueError(f"No processed dataset files found under {PROCESSED_DATA_DIR}")

    env_info = _collect_env_info(dataset_specs)
    GLOBAL_ENV_JSON.write_text(json.dumps(env_info, indent=2), encoding="utf-8")
    print("global env info saved:", GLOBAL_ENV_JSON)

    dataset_payloads = [load_dataset(spec) for spec in dataset_specs]
    for payload in dataset_payloads:
        dataset_paths = _dataset_output_paths(str(payload["name"]))
        dataset_env = {
            "run_tag": RUN_TAG,
            "dataset": payload["name"],
            "dataset_tag": dataset_paths["dataset_tag"],
            "dataset_path": payload["path"],
            "metadata_path": payload.get("metadata_path", ""),
            "dim": int(payload["dim"]),
            "n_total": int(payload["n_total"]),
            "n_train": int(payload["n_train"]),
            "n_test": int(payload["n_test"]),
            "available_arrays": payload.get("available_arrays", []),
            "source_metadata": payload.get("metadata", {}),
            "kernel_specs": KERNEL_SPECS,
            "eps_list": list(EPS_LIST),
            "mode_specs": MODE_SPECS,
            "precompute_methods": PRECOMPUTE_METHODS,
            "reg_lambda": REG_LAMBDA,
            "solve_tol": SOLVE_TOL,
            "gpu_maxiter": GPU_MAXITER,
            "gpu_nufft": GPU_NUFFT,
            "l2_scaled": L2_SCALED,
            "enable_slq": bool(ENABLE_SLQ),
        }
        dataset_paths["env_json"].write_text(json.dumps(dataset_env, indent=2), encoding="utf-8")
        print(
            f"loaded dataset={payload['name']} | dim={payload['dim']} | "
            f"n_total={payload['n_total']} | n_train={payload['n_train']} | n_test={payload['n_test']} | "
            f"dataset_dir={dataset_paths['dataset_dir']}"
        )
else:
    print("Reusing existing dataset_payloads for benchmark / SLQ.")

if "raw_df" not in globals() or "summary_df" not in globals():
    if RUN_WARMUP and len(dataset_payloads) > 0:
        run_warmup(dataset_payloads[0])

    raw_df = run_benchmark(dataset_payloads)
    summary_df, per_dataset_summary = summarize_benchmark(raw_df)
else:
    print("Reusing existing raw_df / summary_df for downstream reporting.")
print("global raw csv:", GLOBAL_RAW_CSV)
print("global summary csv:", GLOBAL_SUMMARY_CSV)
print("raw rows:", len(raw_df))
print("summary rows:", len(summary_df))
print("dataset result dirs:")
for payload in dataset_payloads:
    print("  ", _dataset_output_paths(str(payload["name"]))["dataset_dir"])

if not summary_df.empty:
    display_cols = [
        "dataset",
        "kernel_name",
        "eps",
        "mode",
        "method_variant",
        "eig_method_effective",
        "top_q",
        "precompute_method",
        "nystrom_variant",
        "eigenpro_nystrom_precond_kind_effective",
        "eigenpro_nystrom_refine_mode_effective",
        "eigenpro_nystrom_refine_iters_requested",
        "time_train_median",
        "wall_s_total_median",
        "time_precompute_median",
        "time_eigenspace_median",
        "time_solve_median",
        "cg_iters_median",
        "rmse_test_median",
        "mae_test_median",
        "r2_test_median",
        "peak_mem_gb_median",
        "repeat_count",
        "fail_count",
    ]
    display_cols = [c for c in display_cols if c in summary_df.columns]
    with pd.option_context("display.max_rows", 500, "display.max_columns", 200):
        display(summary_df[display_cols])
else:
    print("No successful runs to summarize.")

slq_df = run_selected_slq(dataset_payloads)
if ENABLE_SLQ:
    print("slq rows:", len(slq_df))
    if not slq_df.empty:
        display(slq_df)

In [ ]:
import json
import shutil
import time
from datetime import timedelta
from pathlib import Path

try:
    from google.colab import files, runtime
    _HAS_COLAB_EXPORT = True
except Exception:
    files = None
    runtime = None
    _HAS_COLAB_EXPORT = False

# --- 1. 结束计时 ---
end_time = time.time()
try:
    elapsed_total = end_time - start_time
    time_str = str(timedelta(seconds=int(elapsed_total)))
except NameError:
    time_str = "未知（未检测到 start_time）"

# --- 2. 解析本次实验需要导出的产物 ---
out_dir_path = Path(OUT_DIR).resolve()
paper_artifacts_dir_eff = globals().get("PAPER_ARTIFACTS_DIR", Path(OUT_DIR) / "paper_artifacts")
paper_dir_path = Path(paper_artifacts_dir_eff).resolve()
notebook_src_path = (Path(BENCHMARK_DIR) / "gpu_benchmark_datasets.ipynb").resolve()

if "DRIVE_OUTPUT_DIR" in globals():
    drive_output_dir = Path(DRIVE_OUTPUT_DIR)
else:
    drive_output_dir = Path(DRIVE_MYDRIVE_DIR) / "EFGP_Eigenpro" / "benchmark_exports"
drive_output_dir.mkdir(parents=True, exist_ok=True)

zip_name = f"{out_dir_path.name}_bundle.zip"
export_root = out_dir_path.parent / f"{out_dir_path.name}_export_bundle"
zip_base = out_dir_path.parent / f"{out_dir_path.name}_bundle"
zip_path = Path(f"{zip_base}.zip")
target_drive_path = drive_output_dir / zip_name

required_checks = {
    "out_dir_exists": out_dir_path.exists(),
    "paper_artifacts_exists": paper_dir_path.exists(),
    "notebook_exists": notebook_src_path.exists(),
}
print("artifact checks:", json.dumps({k: bool(v) for k, v in required_checks.items()}, indent=2))

if not bool(required_checks["out_dir_exists"]):
    raise FileNotFoundError(f"OUT_DIR not found: {out_dir_path}")
if not bool(required_checks["paper_artifacts_exists"]):
    raise FileNotFoundError(f"paper artifacts dir not found: {paper_dir_path}")

# --- 3. 组装 export bundle（只包含本次实验相关内容） ---
if export_root.exists():
    shutil.rmtree(export_root)
export_root.mkdir(parents=True, exist_ok=True)

export_results_dir = export_root / out_dir_path.name
shutil.copytree(out_dir_path, export_results_dir)

notebook_dst_dir = export_root / "notebook"
notebook_dst_dir.mkdir(parents=True, exist_ok=True)
if notebook_src_path.exists():
    shutil.copy2(notebook_src_path, notebook_dst_dir / notebook_src_path.name)

export_manifest = {
    "run_tag": out_dir_path.name,
    "elapsed": time_str,
    "export_root": str(export_root),
    "out_dir": str(out_dir_path),
    "paper_artifacts_dir": str(paper_dir_path),
    "notebook_src": str(notebook_src_path),
    "drive_output_dir": str(drive_output_dir),
    "includes": [
        str(export_results_dir),
        str(notebook_dst_dir / notebook_src_path.name) if notebook_src_path.exists() else "notebook_missing",
    ],
}
(export_root / "export_manifest.json").write_text(json.dumps(export_manifest, indent=2), encoding="utf-8")

# --- 4. 打包 zip ---
if zip_path.exists():
    zip_path.unlink()
print(f"📦 正在打包本次实验目录: {export_root}")
shutil.make_archive(str(zip_base), "zip", root_dir=str(export_root.parent), base_dir=export_root.name)
print("zip saved:", zip_path)

# --- 5. 同步到 Drive 并校验 ---
print(f"💾 正在同步压缩包到 Google Drive: {target_drive_path}")
shutil.copy2(zip_path, target_drive_path)

drive_ok = target_drive_path.exists() and target_drive_path.stat().st_size > 0
local_ok = zip_path.exists() and zip_path.stat().st_size > 0
print("local zip ok:", local_ok)
print("drive zip ok:", drive_ok)

# --- 6. 浏览器下载备份 ---
if local_ok:
    if _HAS_COLAB_EXPORT and files is not None:
        print("📥 启动浏览器备份下载...")
        files.download(str(zip_path))
    else:
        print("当前不在 Colab 环境，跳过浏览器下载。")

print("-" * 30)
print("📊 实验状态:", "成功完成并备份" if drive_ok else "本地打包成功，但 Drive 同步失败")
print(f"⏱️ 累计运行耗时: {time_str}")
print("📁 OUT_DIR:", out_dir_path)
print("📁 Paper artifacts:", paper_dir_path)
print("📦 Local zip:", zip_path)
print("☁️ Drive zip:", target_drive_path)
print("-" * 30)

if drive_ok and _HAS_COLAB_EXPORT and runtime is not None:
    print("等待 20 秒后将自动断开 GPU 以节省点数...")
    time.sleep(20)
    runtime.unassign()
elif not drive_ok:
    print("❌ Drive 同步校验失败；已取消自动断开，请手动检查。")
